# 🎯 Job Interview Guide Workshop — Personalized ML Study Notebook
### CSCN8010 – Machine Learning Frameworks (Weeks 1–7)

This notebook was generated from **two assessments**, grounded in the course study guide and the class repositories
(KNN, Linear Regression Architecture, Logistic Regression, Performance Metrics, and Data Stream Visualization workshops):

| Assessment | Format | Result |
|---|---|---|
| **A. Mock job interview** | 15 open-ended questions (12 technical + 3 behavioral), scored 0–4 each | **21 / 60 (35%)** · Technical 15/48 · Behavioral 6/12 |
| **B. Multiple-choice quiz** (professor's prompt) | 15 MCQs (A–E), technical + scenario/behavioral | **13 / 15 (87%)** |

> **Insight:** you *recognize* correct answers well (MCQ 87%) but find it harder to *explain* them from scratch
> (interview 35%). Interviews reward explanation, so practise saying each module's recap aloud.

### How to use this notebook
1. Read **Section 1** (both recorded assessments) and complete the **Section 2 reflection** cells in your own words.
2. Follow the **study plan** in Section 3 — modules are ordered by priority (biggest gaps first).
3. Each module has: 📘 recap → 🧪 runnable demo → ⚠️ interview trap → 🏋️ exercises (`# TODO` scaffolds) → 💡 collapsible solutions.
4. Run `check(...)` cells to self-grade exercises: ⏳ = not done yet, ✅ = correct, ❌ = try again.
5. Finish with the **12-question re-quiz** in Section 6.

> Runs fully offline: only NumPy, pandas, Matplotlib and scikit-learn's **bundled** datasets
> (`load_iris`, `load_diabetes`, `load_digits`, `load_breast_cancer`) plus synthetic data. No API keys or databases.

### Contents
- [Section 0 – Setup](#setup)
- [Section 1A – Mock interview results](#results)
- [Section 1B – Multiple-choice quiz results](#mcq)
- [Section 2 – Reflection](#reflection)
- [Section 3 – Study plan](#plan)
- **Foundations (missed in the MCQ quiz)**
  - [Module A – Supervised vs unsupervised learning (KNN ≠ K-Means)](#ma) (MCQ 1)
  - [Module B – Variable types & categorical encoding](#mb) (MCQ 2)
- **Priority 1 (scored 0)**
  - [Module 1 – Data leakage, cross-validation & time-series validation](#m1) (Q3, Q10)
  - [Module 2 – Bias–variance & model complexity](#m2) (Q6)
  - [Module 3 – Classification metrics & thresholds](#m3) (Q9)
- **Priority 2 (scored 1)**
  - [Module 4 – Gradient descent & learning rate](#m4) (Q4)
  - [Module 5 – Regression metrics](#m5) (Q5)
  - [Module 6 – Decision tree splits by hand](#m6) (Q7)
- **Priority 3 (scored 2 — refreshers)**
  - [Module 7 – Logistic regression from scratch](#m7) (Q8)
  - [Module 8 – KNN internals, scaling & curse of dimensionality](#m8) (Q1, Q2, Q11)
  - [Module 9 – Streaming residual alerts](#m9) (Q12)
- [Section 5 – Behavioral answers (STAR)](#behavioral) (Q13–Q15)
- [Section 6 – Re-quiz](#requiz)
- [Submission checklist](#submit)

---
<a id="setup"></a>
## Section 0 – Setup
Run this cell first. It imports libraries, fixes the random seed for reproducibility, and defines the `check()` self-grading helper.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_diabetes, load_digits, load_breast_cancer
from sklearn.model_selection import (train_test_split, cross_val_score, cross_validate, cross_val_predict,
                                     GridSearchCV, KFold, StratifiedKFold, TimeSeriesSplit, validation_curve)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import adjusted_rand_score, silhouette_score, pairwise_distances
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report, precision_recall_curve,
                             mean_squared_error, mean_absolute_error, r2_score,
                             mean_absolute_percentage_error)

warnings.filterwarnings("ignore")
RNG = 42
np.random.seed(RNG)
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True


def check(name, fn, expected=None, atol=1e-6):
    """Self-grading helper. `fn` is a zero-arg lambda that runs your code.
    If it returns None (unfinished TODO) you see ⏳; otherwise it is compared to `expected`
    (or, if expected is None, the returned value must be True)."""
    try:
        got = fn()
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet")
        return
    except Exception as exc:  # show the error without stopping the notebook
        print(f"❌ {name}: raised {type(exc).__name__}: {exc}")
        return
    if got is None:
        print(f"⏳ {name}: not implemented yet (returned None)")
        return
    if expected is None:
        ok = bool(got)
    else:
        try:
            ok = np.allclose(np.asarray(got, dtype=float), np.asarray(expected, dtype=float), atol=atol)
        except (TypeError, ValueError):
            ok = got == expected
    print(f"{'✅' if ok else '❌'} {name}: got {got!r}" + ("" if expected is None else f", expected {expected!r}"))


print("Setup complete ✔")

---
<a id="results"></a>
## Section 1A – Recorded Mock Interview Results (open-ended)

**Scoring rubric:** 0 = no answer · 1 = major gaps · 2 = partial · 3 = solid · 4 = interview-ready

| # | Topic | Type | Curriculum area | Score |
|---|---|---|---|---|
| 1 | Parametric vs non-parametric; lazy KNN | Technical | ML Foundations | 2 / 4 |
| 2 | Min-Max vs standardization | Technical | Feature Scaling | 2 / 4 |
| 3 | Data leakage when tuning k | Technical | Validation & Leakage | **0** / 4 |
| 4 | Gradient descent & learning rate | Technical | Optimization | **1** / 4 |
| 5 | R², RMSE vs MAE, MAPE | Technical | Regression Metrics | **1** / 4 |
| 6 | Bias–variance & model complexity | Technical | Bias–Variance | **0** / 4 |
| 7 | Decision tree split by hand | Technical | Decision Trees | **1** / 4 |
| 8 | Logistic regression, sigmoid, log loss | Technical | Logistic Regression | 2 / 4 |
| 9 | Precision, recall & class imbalance | Technical | Classification Metrics | **0** / 4 |
| 10 | Cross-validation & TimeSeriesSplit | Technical | Validation & Leakage | 2 / 4 |
| 11 | KNN scaling & curse of dimensionality | Technical | KNN | 2 / 4 |
| 12 | Streaming residual alerts (Lab 1) | Technical | ML System Design | 2 / 4 |
| 13 | Peer code review (STAR) | Behavioral | Behavioral | 3 / 4 |
| 14 | Leaked credentials (ethics/security) | Behavioral | Behavioral | 2 / 4 |
| 15 | Weak results & stakeholder communication | Behavioral | Behavioral | **1** / 4 |
| | **Total** | | | **21 / 60 (35%)** |

**Pattern:** every **0** sits in *model evaluation* (leakage, bias–variance, classification metrics). Strongest areas were
**feature-scaling intuition for KNN (Q11)**, **streaming system design (Q12)** and **peer code review (Q13)**.

In [ ]:
# Quiz results as a DataFrame (useful for your own analysis / plots)
quiz = pd.DataFrame([
 {
  "q": 1,
  "topic": "Parametric vs non-parametric; lazy KNN",
  "type": "Technical",
  "area": "ML Foundations",
  "score": 2
 },
 {
  "q": 2,
  "topic": "Min-Max vs standardization",
  "type": "Technical",
  "area": "Feature Scaling",
  "score": 2
 },
 {
  "q": 3,
  "topic": "Data leakage when tuning k",
  "type": "Technical",
  "area": "Validation & Leakage",
  "score": 0
 },
 {
  "q": 4,
  "topic": "Gradient descent & learning rate",
  "type": "Technical",
  "area": "Optimization",
  "score": 1
 },
 {
  "q": 5,
  "topic": "R\u00b2, RMSE vs MAE, MAPE",
  "type": "Technical",
  "area": "Regression Metrics",
  "score": 1
 },
 {
  "q": 6,
  "topic": "Bias\u2013variance & model complexity",
  "type": "Technical",
  "area": "Bias\u2013Variance",
  "score": 0
 },
 {
  "q": 7,
  "topic": "Decision tree split by hand",
  "type": "Technical",
  "area": "Decision Trees",
  "score": 1
 },
 {
  "q": 8,
  "topic": "Logistic regression, sigmoid, log loss",
  "type": "Technical",
  "area": "Logistic Regression",
  "score": 2
 },
 {
  "q": 9,
  "topic": "Precision, recall & class imbalance",
  "type": "Technical",
  "area": "Classification Metrics",
  "score": 0
 },
 {
  "q": 10,
  "topic": "Cross-validation & TimeSeriesSplit",
  "type": "Technical",
  "area": "Validation & Leakage",
  "score": 2
 },
 {
  "q": 11,
  "topic": "KNN scaling & curse of dimensionality",
  "type": "Technical",
  "area": "KNN",
  "score": 2
 },
 {
  "q": 12,
  "topic": "Streaming residual alerts (Lab 1)",
  "type": "Technical",
  "area": "ML System Design",
  "score": 2
 },
 {
  "q": 13,
  "topic": "Peer code review (STAR)",
  "type": "Behavioral",
  "area": "Behavioral",
  "score": 3
 },
 {
  "q": 14,
  "topic": "Leaked credentials (ethics/security)",
  "type": "Behavioral",
  "area": "Behavioral",
  "score": 2
 },
 {
  "q": 15,
  "topic": "Weak results & stakeholder communication",
  "type": "Behavioral",
  "area": "Behavioral",
  "score": 1
 }
])
quiz["pct"] = quiz["score"] / 4 * 100

print(f"Total: {quiz.score.sum()} / {4 * len(quiz)}  ({quiz.score.sum() / (4 * len(quiz)):.0%})")
display(quiz.groupby("type")["score"].agg(["sum", "count", "mean"]).rename(columns={"mean": "avg / 4"}))

colors = quiz["score"].map({0: "#c0392b", 1: "#e67e22", 2: "#f1c40f", 3: "#27ae60", 4: "#1e8449"})
fig, ax = plt.subplots(figsize=(10, 5))
labels = quiz["q"].astype(str) + "  " + quiz["topic"]
ax.barh(labels, quiz["score"].clip(lower=0.05), color=colors)   # tiny bar so 0-scores stay visible
for i, s in enumerate(quiz["score"]):
    ax.text(s + 0.05, i, str(s), va="center", fontweight="bold")
ax.invert_yaxis()
ax.set_xlim(0, 4)
ax.set_xlabel("Score (0–4)")
ax.set_title("Mock interview score per question")
plt.tight_layout()
plt.show()

### Detailed question-by-question record
Your answers are recorded verbatim. Feedback and the key points of a model answer follow each one.

#### Q1 · Parametric vs non-parametric; lazy KNN · *Technical* · **Score 2/4**
**Question:** Classify linear regression, logistic regression, decision trees and KNN as parametric or non-parametric and explain the difference. What does 'lazy learner' mean for KNN, and what does `knn.fit()` actually do?

**Your answer (verbatim):**
> Parametric: Linear and Logistic regression. Non-parametric: Decision and KNN
>
> KNN does not learn at the beginning only when the model is trying to predict. 
> Training time is fast but the prediction time is slow.
> I am not sure about the knn.fit.

- ✅ **What worked:** Correct classification of all four models; correct intuition for lazy learning (fast fit, slow predict).
- 🔧 **Gap:** Did not define *why* a model is parametric (fixed number of parameters vs complexity that grows with data). Did not know that `fit()` just stores the training data (and optionally builds a KD/Ball tree).

**Model-answer key points:**
- Parametric = fixed number of learned parameters; training data can be discarded after fitting.
- Non-parametric = model complexity grows with the data (tree structure, stored KNN instances).
- `KNeighborsClassifier.fit()` stores X/y (+ optional search index). All distance work happens at `predict()` → O(n·d) per query.

#### Q2 · Min-Max vs standardization · *Technical* · **Score 2/4**
**Question:** Write the Min-Max formula including a custom range [a, b]. What happens for a constant feature? How does one large outlier affect Min-Max vs StandardScaler?

**Your answer (verbatim):**
> Min-max norm = (x-xmin)/(xmax - xmin). Not sure about a custom range.
>
> If the value is constant, the normalized value would undefined as you would get 0/0. I think this feature should be excluded.
>
> The resulting normalized values would be very small for most of the data while the outlier will have a value of 1. Not sure how standardization works ...

- ✅ **What worked:** Correct base formula, identified the 0/0 problem, correct description of outlier compression.
- 🔧 **Gap:** Custom-range formula and standardization (z-score) were missing.

**Model-answer key points:**
- x' = a + (x − min)(b − a)/(max − min); `MinMaxScaler(feature_range=(a, b))`.
- Constant feature: guard `max == min` → drop it or map to `a` (sklearn maps to 0, never NaN).
- Standardization z = (x − μ)/σ → mean 0, std 1, unbounded; less squashing from outliers; RobustScaler (median/IQR) is best for outliers.

#### Q3 · Data leakage when tuning k · *Technical* · **Score 0/4**
**Question:** Your Titanic notebook chooses `best_k` by looping k = 1..25 and scoring on `X_test`. What is wrong, how do you fix it, and why do the imputer/scaler belong inside the Pipeline?

**Your answer (verbatim):**
> I dont remember this.

- ✅ **What worked:** —
- 🔧 **Gap:** Did not recognize tuning on the test set as leakage / optimistic bias.

**Model-answer key points:**
- Selecting hyperparameters with test scores turns the test set into a validation set → optimistic, non-generalizable estimate.
- Fix: `GridSearchCV(pipeline, {'classifier__n_neighbors': range(1, 26)}, cv=5)` on train only; score test once at the end.
- Preprocessing inside the Pipeline is re-fit on each CV training fold, so validation/test rows never influence medians, means or stds.

#### Q4 · Gradient descent & learning rate · *Technical* · **Score 1/4**
**Question:** Given the scratch GD loop in `src/model.py`: which cost function produces these gradients and why does `grad_theta_1` have `* X_train`? What happens with a learning rate that is too large / too small? Batch vs SGD vs mini-batch?

**Your answer (verbatim):**
> The cost function measures how wrong a model is.
>
> Maybe learning rate if too large causes over fitting. Not sure.
>
> I think SGD was using a single value to calculate and then continues on one by one. Mini-batch takes a small sample size.

- ✅ **What worked:** SGD (one sample per update) and mini-batch (small subset) are described correctly.
- 🔧 **Gap:** Did not name MSE or derive the chain-rule term. Misconception: a large learning rate does **not** cause overfitting — it causes oscillation/divergence.

**Model-answer key points:**
- J = (1/2m)Σ(ŷ − y)²; ∂J/∂θ₀ = mean(error); ∂J/∂θ₁ = mean(error·x) because ∂ŷ/∂θ₁ = x.
- LR too large → loss oscillates / explodes to inf/nan. Too small → very slow convergence. Diagnose with a loss-vs-epoch plot.
- Batch = all m rows/update (smooth, costly); SGD = 1 row (noisy, cheap); mini-batch = 32–256 rows (practical default).

#### Q5 · R², RMSE vs MAE, MAPE · *Technical* · **Score 1/4**
**Question:** Interpret R² = 0.4589 (California, MedInc only). Why is RMSE ≥ MAE and what does a large gap mean? When does MAPE break?

**Your answer (verbatim):**
> Measure the size of the variance I think. 1 is perfect predictions with zero variance. So this model would be okay but not great.
>
> I am not sure on this one.
>
> When values are 0.

- ✅ **What worked:** Reasonable overall verdict; correctly identified MAPE fails at y = 0.
- 🔧 **Gap:** R² definition imprecise (it is the *proportion of variance explained* vs a mean baseline). RMSE vs MAE not answered.

**Model-answer key points:**
- R² = 1 − SS_res/SS_tot: model explains ~46% of variance vs predicting the mean; can be negative on test data.
- RMSE squares errors → penalizes large errors; RMSE ≥ MAE always; a large gap ⇒ a few big errors / outliers.
- MAPE: undefined at y = 0, explodes near 0, asymmetric (biased toward under-prediction).

#### Q6 · Bias–variance & model complexity · *Technical* · **Score 0/4**
**Question:** Polynomial degree 0→5 on BMI (diabetes): what happens to training vs validation error? Relate to bias–variance. What is the equivalent complexity knob for KNN and decision trees?

**Your answer (verbatim):**
> Not sure how to answer this question.

- ✅ **What worked:** —
- 🔧 **Gap:** Bias–variance trade-off and train/validation error curves not yet internalized.

**Model-answer key points:**
- Training error ↓ monotonically with degree; validation error is U-shaped. Degree 0 underfits, high degree overfits.
- Underfit = high bias (both errors high, small gap). Overfit = high variance (low train, high validation, large gap).
- KNN: small k overfits, large k underfits. Tree: deep / tiny leaves overfits, shallow underfits.

#### Q7 · Decision tree split by hand · *Technical* · **Score 1/4**
**Question:** Data x = [1,2,3,4], y = [10,12,30,32]. How does a regression tree pick the threshold? Which threshold wins and what are the leaf predictions? How do classification trees differ?

**Your answer (verbatim):**
> I think it was the MSE it computes and takes the one that is the lowest.
>
> I am not sure on the rest.

- ✅ **What worked:** Correct core idea: compute MSE and choose the minimum.
- 🔧 **Gap:** Did not mention midpoint candidates, sample-weighted child MSE, the worked result, or Gini/entropy.

**Model-answer key points:**
- Candidates = midpoints 1.5, 2.5, 3.5. Score = (n_L·MSE_L + n_R·MSE_R)/n.
- Weighted MSE: 60.67, **1.0**, 60.67 → split at x ≤ 2.5; leaves predict means 11 and 31 (parent MSE 101).
- Classification trees use Gini = 1 − Σp² or entropy; leaves predict the majority class.

#### Q8 · Logistic regression, sigmoid, log loss · *Technical* · **Score 2/4**
**Question:** Why is linear regression a poor fit for pass/fail? Write the sigmoid and explain how its output becomes a class. What loss does logistic regression use and why?

**Your answer (verbatim):**
> It has something I think to do with a straight line not representing the predictions correctly.
>
> I cant remember the sigmoid function.
>
> I think it uses Log Loss to ensure a convex curve.

- ✅ **What worked:** Named log loss and gave the convexity reason — a strong point.
- 🔧 **Gap:** Vague critique of linear regression; sigmoid formula not recalled.

**Model-answer key points:**
- Linear regression outputs unbounded values (not probabilities) and is pulled by extreme x values.
- σ(z) = 1/(1 + e^(−z)), z = w·x + b → P(y=1|x). Predict 1 if p ≥ 0.5 ⇔ z ≥ 0 (linear boundary).
- BCE = −mean[y·log p + (1−y)·log(1−p)]: convex, maximum likelihood; gradient = Xᵀ(p − y)/m.

#### Q9 · Precision, recall & class imbalance · *Technical* · **Score 0/4**
**Question:** Why does a DummyClassifier get ~90% accuracy on 'Is this a 5?' (MNIST)? Define precision and recall with TP/FP/FN. For heart-disease screening, which metric matters most and how do you push toward it?

**Your answer (verbatim):**
> I cant remember exactly.

- ✅ **What worked:** —
- 🔧 **Gap:** Accuracy paradox, confusion-matrix definitions and threshold tuning all need review.

**Model-answer key points:**
- ~10% of digits are 5s → always predicting 'not 5' yields ~90% accuracy and zero recall. Accuracy misleads under imbalance.
- Precision = TP/(TP+FP) ('when I say yes, am I right?'). Recall = TP/(TP+FN) ('did I catch them all?'). F1 = harmonic mean.
- Screening → prioritize recall: lower the threshold on predict_proba, class_weight='balanced', tune with scoring='recall'.

#### Q10 · Cross-validation & TimeSeriesSplit · *Technical* · **Score 2/4**
**Question:** Explain 5-fold CV: how many models, what do you report? Why is it better than one split on a 442-row dataset? Why is shuffled K-fold wrong for time-series robot telemetry?

**Your answer (verbatim):**
> A 5-fold cross validation splits the train data into 5 equal parts. Parts 1,2,3,4 get used for training and part 5 for validation. This changes until all parts were used for validation. So I would say 5 models get trained. I cant remember what number it reported at the end.
>
> On a small data set, you may not cover all the variances of the data in one split, especially on a small dataset where you dont have that many values to begin with.
>
> Not sure about the last part.

- ✅ **What worked:** Correct fold rotation, 5 models, good intuition about small datasets.
- 🔧 **Gap:** Report mean ± std across folds; time-series requires forward-chaining validation (TimeSeriesSplit).

**Model-answer key points:**
- Report mean ± std of the k fold scores; refit the chosen model on all training data afterwards.
- Every row is validated once → lower-variance estimate than a single 88-row test split.
- Shuffled K-fold trains on the future → leakage. Use `TimeSeriesSplit` / chronological holdout.

#### Q11 · KNN scaling & curse of dimensionality · *Technical* · **Score 2/4**
**Question:** What happens to KNN distances if Fare (0–512) is unscaled next to SibSp (0–8)? What is the curse of dimensionality? Give two situations where you would not use KNN.

**Your answer (verbatim):**
> KNN is sensitive to the distances. With Fare having a large scale and range it influences the model more than SIbsp would with a smaller scale and range. For KNN data would have to be normalized using a scaling technique for all features to offer fair influence.
>
> I remember the term but not the explanation.
>
> Not sure about number 3\

- ✅ **What worked:** Excellent, interview-ready explanation of distance dominance by large-scale features.
- 🔧 **Gap:** Curse of dimensionality and 'when not to use KNN' missing.

**Model-answer key points:**
- High dimensions → sparse data and distance concentration (nearest ≈ farthest) → neighbours stop being meaningful.
- Avoid KNN for: large n / low-latency serving, high-dimensional sparse data, need for interpretability, extrapolation.

#### Q12 · Streaming residual alerts (Lab 1) · *Technical* · **Score 2/4**
**Question:** How does linear regression + residuals drive Alert vs Error in the robot predictive-maintenance pipeline? Why a duration condition? Where do Neon/PostgreSQL and `.env` fit?

**Your answer (verbatim):**
> I cant remember the first part as that was a while ago.
>
> The threshold was at which point the normal readings would be considered abnormal. A duration was added as a single reading above the threshold would not be considered abornal operation but a sustained duration above the threshold would be.
>
> The database was used to simulate real world MLOPs as the model would be pulling data live directly from a database which would collect the readings from the robot. This would happen every so many seconds, so you would have a live monitoring dashboard updated as the readings are taken.

- ✅ **What worked:** Clear reasoning for duration (filters transient spikes) and good architecture description.
- 🔧 **Gap:** Regression → residual logic and `.env`/secrets role were missing.

**Model-answer key points:**
- Fit regression on healthy data → expected current; residual = actual − predicted.
- Alert ≈ residual > μ+2σ sustained ≥ T s; Error ≈ > μ+3σ sustained. Duration trades recall for precision.
- Secrets (NEON_DATABASE_URL) live in `.env`, loaded via os.getenv / python-dotenv, and `.env` is in `.gitignore`.

#### Q13 · Peer code review (STAR) · *Behavioral* · **Score 3/4**
**Question:** Tell me about a time you found a problem in someone else's code (or someone found one in yours).

**Your answer (verbatim):**
> We were doing peer programming working between groups. We were required to do load the data, perform EDA and then do some preprocessing on the data. When check my peer's code, I found that that EDA was being called before the dataset was properly loaded into the dataframe. I noted this down and check on other sections of the notebook. When I did my review with my peer, I explained the issue and validated it by showing the notebook. Once he realized the issue, he went to work on fixing it and then produced an updated notebook that ran smoothly.

- ✅ **What worked:** Solid STAR structure, specific technical issue, evidence-based feedback, clear result.
- 🔧 **Gap:** Add *how* you framed it tactfully, *why* it mattered (reproducibility / hidden kernel state) and a lasting process change.

**Model-answer key points:**
- Mention 'Restart Kernel & Run All' as a team checklist item — shows learning that outlives the incident.

#### Q14 · Leaked credentials (ethics/security) · *Behavioral* · **Score 2/4**
**Question:** A teammate committed `.env` with NEON_DATABASE_URL to a public repo the night before the deadline and is unreachable. What do you do?

**Your answer (verbatim):**
> This is an issue since trying to uncommit and remove the history can cause syncing issues and affect the repo and branches. Uncommitting will remove the file but the history remains which shows the line of code. One way could be changing the repo to private if there are multiple branches and people working them.
> Another would be to copy all the files and delete the repo, recreating a new one without the .env file commited.
>
> I would try framing it as a question to start the topic. Maybe ask something about share passwords in any files etc. Then go onto how we usually handle those which is by rather having a .gitgnore file first that keeps everything out of the git repo that we dont want to be pushed.

- ✅ **What worked:** Knows git history retains the secret; constructive, non-blaming follow-up; .gitignore prevention.
- 🔧 **Gap:** Critical miss: **rotate the credential first** — once public, assume compromised. Also notify team/instructor.

**Model-answer key points:**
- Order: rotate password → check access logs → remove from tracking + purge history → notify → blameless follow-up + pre-commit secret scanning.

#### Q15 · Weak results & stakeholder communication · *Behavioral* · **Score 1/4**
**Question:** Tell me about a model result weaker than expected. How did you diagnose it, what did you try, and how would you explain it to a non-technical stakeholder?

**Your answer (verbatim):**
> In one of our workshop were tasked at looking at housing datasets and using various feature to predict housing prices. Since we had many features we did normalization and used PCA to determine what features would best helps us in our traget predictions. However, we found that all the various features had very weak relationships to the housing price.

- ✅ **What worked:** Relevant example and a reasonable first step.
- 🔧 **Gap:** Answer stops at the situation: no diagnosis, iteration, result or stakeholder explanation. PCA creates combinations of features; it does not select original features.

**Model-answer key points:**
- Structure: S/T → diagnose (correlations, residuals, baseline) → iterate (features, models, CV) → result → plain-language limits + next step.

---
<a id="mcq"></a>
## Section 1B – Recorded Multiple-Choice Quiz Results

Professor-provided prompt: 15 MCQs (A–E) covering supervised vs unsupervised, variables, train/val/test, linear & logistic
regression (R², MSE, cross-entropy), KNN hyperparameters, decision-tree leaves, and scenario/behavioral items.
Questions were asked one at a time and scored at the end.

| # | Topic | Your answer | Correct | Result |
|---|---|---|---|---|
| 1 | Supervised vs unsupervised | B | C | ❌ |
| 2 | Variable types & encoding | A | C | ❌ |
| 3 | Train / validation / test | B | B | ✅ |
| 4 | R² | C | C | ✅ |
| 5 | MSE & outliers | C | C | ✅ |
| 6 | Cross-entropy | E | E | ✅ |
| 7 | Logistic output & threshold | B | B | ✅ |
| 8 | KNN `weights` | B | B | ✅ |
| 9 | KNN distance `p` | B | B | ✅ |
| 10 | Choosing k | C | C | ✅ |
| 11 | Tree leaf prediction | C | C | ✅ |
| 12 | Tree hyperparameters | C | C | ✅ |
| 13 | Imbalanced data | B | B | ✅ |
| 14 | Data leakage | C | C | ✅ |
| 15 | Model-choice trade-offs | B | B | ✅ |
| | **Total** | | | **13 / 15 (87%)** |

In [ ]:
mcq = pd.DataFrame([
 {
  "q": 1,
  "topic": "Supervised vs unsupervised",
  "yours": "B",
  "correct": "C"
 },
 {
  "q": 2,
  "topic": "Variable types & encoding",
  "yours": "A",
  "correct": "C"
 },
 {
  "q": 3,
  "topic": "Train / validation / test",
  "yours": "B",
  "correct": "B"
 },
 {
  "q": 4,
  "topic": "R²",
  "yours": "C",
  "correct": "C"
 },
 {
  "q": 5,
  "topic": "MSE & outliers",
  "yours": "C",
  "correct": "C"
 },
 {
  "q": 6,
  "topic": "Cross-entropy",
  "yours": "E",
  "correct": "E"
 },
 {
  "q": 7,
  "topic": "Logistic output & threshold",
  "yours": "B",
  "correct": "B"
 },
 {
  "q": 8,
  "topic": "KNN `weights`",
  "yours": "B",
  "correct": "B"
 },
 {
  "q": 9,
  "topic": "KNN distance `p`",
  "yours": "B",
  "correct": "B"
 },
 {
  "q": 10,
  "topic": "Choosing k",
  "yours": "C",
  "correct": "C"
 },
 {
  "q": 11,
  "topic": "Tree leaf prediction",
  "yours": "C",
  "correct": "C"
 },
 {
  "q": 12,
  "topic": "Tree hyperparameters",
  "yours": "C",
  "correct": "C"
 },
 {
  "q": 13,
  "topic": "Imbalanced data",
  "yours": "B",
  "correct": "B"
 },
 {
  "q": 14,
  "topic": "Data leakage",
  "yours": "C",
  "correct": "C"
 },
 {
  "q": 15,
  "topic": "Model-choice trade-offs",
  "yours": "B",
  "correct": "B"
 }
])
mcq["is_correct"] = mcq["yours"] == mcq["correct"]
print(f"MCQ score: {mcq.is_correct.sum()} / {len(mcq)}  ({mcq.is_correct.mean():.0%})")
display(mcq[~mcq.is_correct])

summary = pd.DataFrame({
    "assessment": ["Mock interview (explain)", "MCQ (recognize)"],
    "score_pct": [quiz.score.sum() / (4 * len(quiz)) * 100, mcq.is_correct.mean() * 100],
})
ax = summary.plot.bar(x="assessment", y="score_pct", legend=False, color=["#e67e22", "#27ae60"], rot=0, figsize=(6, 3.5))
ax.set_ylim(0, 100); ax.set_ylabel("% score"); ax.set_title("Recognizing vs explaining")
for i, v in enumerate(summary.score_pct):
    ax.text(i, v + 2, f"{v:.0f}%", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

### Missed questions in detail

#### MCQ 1 · Supervised vs unsupervised ❌
**Question:** 2 years of customer purchase histories with **no labels**; marketing wants to find natural groups of similar customers. Best approach?

- **A.** Logistic regression predicting each customer's group
- **B.** KNN classification with k = 5  ⬅ your answer
- **C.** K-Means clustering  ✅ correct
- **D.** Linear regression on total spend
- **E.** Decision tree classifier on purchase history

**Why:** No labels ⇒ unsupervised. KNN *classification* needs labelled neighbours; K-Means *clustering* discovers groups. Same letter 'k', completely different jobs.

#### MCQ 2 · Variable types & encoding ❌
**Question:** Titanic `Embarked` ∈ {C, Q, S}. Most appropriate treatment before KNN?

- **A.** Map to 1, 2, 3 and standardize  ⬅ your answer
- **B.** Treat it as the target
- **C.** One-hot encode (nominal, no order)  ✅ correct
- **D.** Drop it; KNN can't use categories
- **E.** Min-Max scale the raw letters

**Why:** Integer codes invent an order and distances (C→S looks twice as far as C→Q). Nominal ⇒ one-hot; ordinal (e.g. Pclass, small/medium/large) ⇒ ordered codes.

### Correct answers: one-line takeaways to keep
- **MCQ 3 (Train / validation / test):** Test is touched once; tuning uses validation or CV.
- **MCQ 4 (R²):** R² = 1 − SS_res/SS_tot.
- **MCQ 5 (MSE & outliers):** X: 400 (k$²) vs Y: 2000 (k$²).
- **MCQ 6 (Cross-entropy):** −log(0.01) ≈ 4.6: confident and wrong is punished hardest.
- **MCQ 7 (Logistic output & threshold):** Default threshold 0.5; output is P(y = 1 | x).
- **MCQ 8 (KNN `weights`):** Inverse-distance weighting of votes / averages.
- **MCQ 9 (KNN distance `p`):** p = 1 Manhattan, p = 2 Euclidean (default), p = ∞ Chebyshev.
- **MCQ 10 (Choosing k):** Small k = high variance, large k = high bias.
- **MCQ 11 (Tree leaf prediction):** Regression leaves predict the mean (minimizes MSE).
- **MCQ 12 (Tree hyperparameters):** Overfitting → reduce complexity; trees don't need scaling.
- **MCQ 13 (Imbalanced data):** Accuracy paradox: a dummy also scores 99.5%.
- **MCQ 14 (Data leakage):** Target leakage (future information) + preprocessing leakage.
- **MCQ 15 (Model-choice trade-offs):** Latency + interpretability outweigh 1% F1; communicate with numbers.

---
<a id="reflection"></a>
## Section 2 – Reflection ✍️
*Complete these cells in your own words. Replace each “Your answer here” line.*

**R1. Overall:** How do you feel about your 21/60 result? Which score surprised you most, and why?

> ✍️ *Your answer here*

**R2. Pattern:** All three zero scores were about *evaluating* models (leakage, bias–variance, metrics). Why do you think evaluation is weaker for you than building models?

> ✍️ *Your answer here*

**R3. Misconception check:** In Q4 you suggested a large learning rate causes overfitting. After Module 4, explain in your own words what actually happens and why.

> ✍️ *Your answer here*

**R4. Strengths:** Which answers are you proud of (e.g., Q11 scaling, Q12 duration thresholds, Q13 code review)? What made them strong?

> ✍️ *Your answer here*

**R5. Behavioral:** After Section 5, how would you now re-answer Q14 (leaked credentials)? What is the first action and why?

> ✍️ *Your answer here*

**R6. Action plan:** List the three modules you will revisit first, how much time you will spend, and how you will know you are interview-ready.

> ✍️ *Your answer here*

**R7. After the re-quiz (Section 6):** What is your new score, and what still needs work?

> ✍️ *Your answer here*

**R8. MCQ vs interview:** You scored much higher on the multiple-choice quiz than in the open-ended interview. What does that tell you about how you should study for a real interview?

> ✍️ *Your answer here*

**R9. MCQ misses:** In your own words, explain the difference between KNN and K-Means, and why `Embarked` should not be encoded as 1/2/3.

> ✍️ *Your answer here*

---
<a id="plan"></a>
## Section 3 – Personalized Study Plan

| Priority | Module | Quiz Qs | Score | Suggested time | Goal: you can… |
|---|---|---|---|---|---|
| 🔴 0 | A Supervised vs unsupervised | MCQ 1 | ❌ | 30 min | tell KNN from K-Means and pick the learning type from a problem statement |
| 🔴 0 | B Variable types & encoding | MCQ 2 | ❌ | 30 min | classify each column's type and build a `ColumnTransformer` |
| 🔴 1 | M1 Leakage & validation | Q3, Q10 | 0, 2 | 90 min | spot test-set tuning, write `GridSearchCV` + Pipeline, use `TimeSeriesSplit` |
| 🔴 1 | M2 Bias–variance | Q6 | 0 | 60 min | sketch train vs validation curves and name each model's complexity knob |
| 🔴 1 | M3 Classification metrics | Q9 | 0 | 75 min | compute precision/recall by hand and tune a threshold for recall |
| 🟠 2 | M4 Gradient descent | Q4 | 1 | 60 min | derive the MSE gradient and diagnose learning-rate problems from a loss plot |
| 🟠 2 | M5 Regression metrics | Q5 | 1 | 45 min | define R² precisely and explain RMSE vs MAE vs MAPE |
| 🟠 2 | M6 Tree splits | Q7 | 1 | 45 min | compute the best split by hand (MSE and Gini) |
| 🟡 3 | M7 Logistic regression | Q8 | 2 | 40 min | write sigmoid + BCE from memory |
| 🟡 3 | M8 KNN / scaling | Q1, Q2, Q11 | 2 | 40 min | explain `fit()`, scaling formulas and the curse of dimensionality |
| 🟡 3 | M9 Streaming alerts | Q12 | 2 | 30 min | explain residual thresholds + duration |
| 🟢 | Behavioral STAR | Q13–Q15 | 3, 2, 1 | 45 min | deliver full STAR answers including result + learning |

**Suggested schedule (≈ 10 h):** Day 1 → A, B, M1, M2 · Day 2 → M3, M4 · Day 3 → M5, M6, M7 · Day 4 → M8, M9, Behavioral, Re-quiz.

**Interview-ready test:** you can explain each module's ⚠️ *trap* out loud in under 60 seconds without notes.

---
<a id="ma"></a>
# 🔴 Module A – Supervised vs Unsupervised Learning: KNN ≠ K-Means (MCQ 1)

### 📘 Recap
| | **Supervised** | **Unsupervised** |
|---|---|---|
| Data | features **X and labels y** | features **X only** |
| Goal | predict y for new X | discover structure: groups, low-dimensional views, anomalies |
| Tasks | regression (price, progression) · classification (pass/fail, species) | clustering · dimensionality reduction · anomaly detection |
| Course models | linear / polynomial / logistic regression, decision trees, **KNN** | **K-Means**, PCA (you used PCA in the housing workshop) |
| Evaluation | compare predictions with true labels (R², F1…) | no ground truth: inertia, silhouette score, domain review |

### KNN vs K-Means (the classic trap)
| | **K-Nearest Neighbors** | **K-Means** |
|---|---|---|
| Type | supervised (classification/regression) | unsupervised (clustering) |
| What *k* means | number of **neighbours** that vote | number of **clusters** to find |
| Needs labels? | **yes** | **no** |
| `fit()` does | stores the labelled data (lazy) | iteratively moves *k* centroids (eager) |
| Output | a label/value for a query point | a cluster id for every point + centroids |
| Shared trait | distance-based → **both need feature scaling** | |

**Decision rule:** *"Is there a column I'm trying to predict, with known answers in my training data?"* Yes → supervised. No → unsupervised.

### 🧪 Demo A1 – K-Means finds the Iris species *without ever seeing the labels*

In [ ]:
iris = load_iris()
X_ir = StandardScaler().fit_transform(iris.data)          # distance-based → scale first

km = KMeans(n_clusters=3, n_init=10, random_state=RNG).fit(X_ir)   # note: fit(X) only, no y!
print("Cluster sizes:", np.bincount(km.labels_))
print(f"Agreement with true species (Adjusted Rand Index, 1 = perfect): {adjusted_rand_score(iris.target, km.labels_):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(iris.data[:, 2], iris.data[:, 3], c=iris.target, cmap="viridis", edgecolor="k", s=35)
axes[0].set_title("Truth: species labels (what KNN would need)")
axes[1].scatter(iris.data[:, 2], iris.data[:, 3], c=km.labels_, cmap="tab10", edgecolor="k", s=35)
axes[1].set_title("K-Means clusters (found without labels)")
for ax in axes:
    ax.set_xlabel("petal length (cm)"); ax.set_ylabel("petal width (cm)")
plt.tight_layout(); plt.show()

### 🧪 Demo A2 – Choosing the number of clusters (no labels → elbow and silhouette)

In [ ]:
ks = range(2, 9)
inertias, sils = [], []
for k in ks:
    m = KMeans(n_clusters=k, n_init=10, random_state=RNG).fit(X_ir)
    inertias.append(m.inertia_)
    sils.append(silhouette_score(X_ir, m.labels_))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(list(ks), inertias, "o-"); axes[0].set_title("Elbow: inertia (within-cluster SSE)"); axes[0].set_xlabel("k clusters")
axes[1].plot(list(ks), sils, "s-", c="tab:green"); axes[1].set_title("Silhouette (higher = better separated)"); axes[1].set_xlabel("k clusters")
plt.tight_layout(); plt.show()
print("Best k by silhouette:", list(ks)[int(np.argmax(sils))], "(versicolor & virginica overlap, so 2 often wins; domain knowledge says 3)")

### ⚠️ Interview traps
- "Segment customers with KNN" → wrong tool. KNN needs labels. **Clustering** (K-Means) segments unlabelled data.
- K-Means `fit(X)` has **no y**. If you're passing labels, you're doing supervised learning.
- Both KNN and K-Means use Euclidean distance, so **scale features** for both.
- An unsupervised result has no "accuracy". Talk about inertia, silhouette and whether the clusters make business sense.
- A common hybrid is to cluster first (unsupervised), then use the cluster id as a feature in a supervised model.

### 🏋️ Exercise A.1 – Supervised or unsupervised?
Fill in each value with `'supervised'` or `'unsupervised'`.

In [ ]:
scenarios = {
    "Predict diabetes progression from BMI": None,
    "Group robots with similar current-draw patterns, no failure labels available": None,
    "Predict whether a Titanic passenger survived": None,
    "Reduce 64 pixel features of digits to 2 dimensions for plotting (PCA)": None,
    "Flag unusual sensor readings when no faults have ever been labelled": None,
    "Classify iris species from petal measurements": None,
}
# TODO: replace every None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
scenarios = {
    "Predict diabetes progression from BMI": "supervised",
    "Group robots with similar current-draw patterns, no failure labels available": "unsupervised",
    "Predict whether a Titanic passenger survived": "supervised",
    "Reduce 64 pixel features of digits to 2 dimensions for plotting (PCA)": "unsupervised",
    "Flag unusual sensor readings when no faults have ever been labelled": "unsupervised",
    "Classify iris species from petal measurements": "supervised",
}
```

</details>

In [ ]:
_key = ["supervised", "unsupervised", "supervised", "unsupervised", "unsupervised", "supervised"]
check("Ex A.1 all scenarios", lambda: None if None in scenarios.values() else list(scenarios.values()), _key)

### 🏋️ Exercise A.2 – K-Means from scratch
Implement Lloyd's algorithm: (1) start from the given centroids, (2) **assign** each point to its nearest centroid, (3) **update** each centroid to the mean of its points, (4) repeat until nothing changes.

In [ ]:
def kmeans_scratch(X, init_centroids, n_iter=100):
    """Return (labels, centroids). init_centroids: array of shape (k, n_features)."""
    centroids, k = init_centroids.copy(), len(init_centroids)
    # TODO: loop n_iter times:
    #   distances = pairwise_distances(X, centroids)   -> shape (n, k)
    #   labels = argmin over axis 1
    #   new_centroids = mean of X[labels == j] for each j
    #   stop early if centroids don't change (np.allclose)
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def kmeans_scratch(X, init_centroids, n_iter=100):
    """Return (labels, centroids). init_centroids: array of shape (k, n_features)."""
    centroids, k = init_centroids.copy(), len(init_centroids)
    for _ in range(n_iter):
        labels = pairwise_distances(X, centroids).argmin(axis=1)
        new = np.array([X[labels == j].mean(axis=0) if np.any(labels == j) else centroids[j] for j in range(k)])
        if np.allclose(new, centroids):
            break
        centroids = new
    return labels, centroids
```

</details>

In [ ]:
init = X_ir[np.random.default_rng(RNG).choice(len(X_ir), size=3, replace=False)]   # same starting centroids for both
res_km = kmeans_scratch(X_ir, init)
ref_km = KMeans(n_clusters=3, init=init, n_init=1).fit(X_ir)
check("Ex A.2 matches sklearn KMeans from the same start (ARI = 1)",
      lambda: None if res_km is None else adjusted_rand_score(ref_km.labels_, res_km[0]), 1.0)

---
<a id="mb"></a>
# 🔴 Module B – Variable Types & Categorical Encoding (MCQ 2)

### 📘 Recap: variable types
| Type | Sub-type | Titanic example | Typical treatment |
|---|---|---|---|
| **Target** (label, y) | binary | `Survived` | never a feature; never scaled for classification |
| **Numerical** | continuous | `Age`, `Fare` | impute (median) → **scale** |
| | discrete / count | `SibSp`, `Parch` | impute → scale |
| **Categorical** | **nominal** (no order) | `Sex`, `Embarked` | **one-hot encode** |
| | **ordinal** (real order) | `Pclass` (1st > 2nd > 3rd), small/medium/large | ordered integer codes (`OrdinalEncoder` with explicit order) → scale |
| **Identifier** | — | `PassengerId`, `Name`, `Ticket` | drop (or engineer a feature, e.g. title from `Name`) |

### Why label-encoding a nominal variable is wrong for KNN
With `C=1, Q=2, S=3`: distance(C, Q) = 1, distance(C, S) = **2**. The model now "believes" Cherbourg is twice as different from
Southampton as from Queenstown, a made-up geometry. One-hot vectors make every pair of different ports **equally** far apart (√2).

> Tree models are less sensitive to this (they split on thresholds), but one-hot is still the safe default for nominal features.
> Linear models and KNN are directly distorted by fake ordering.

### 🧪 Demo B1 – The fake geometry of integer codes vs one-hot

In [ ]:
ports = np.array([["C"], ["Q"], ["S"]])
label_codes = np.array([[1], [2], [3]])
onehot = OneHotEncoder().fit_transform(ports).toarray()

show = lambda D: pd.DataFrame(D, index=["C", "Q", "S"], columns=["C", "Q", "S"]).round(2)
print("Distances with label encoding 1/2/3:")
display(show(pairwise_distances(label_codes)))
print("Distances with one-hot encoding:")
display(show(pairwise_distances(onehot)))

### 🧪 Demo B2 – A Titanic-style mixed-type dataset through a `ColumnTransformer`
The Titanic CSV isn't bundled with scikit-learn, so we generate a small realistic stand-in (same column names and types, with missing values).

In [ ]:
rng = np.random.default_rng(RNG)
n = 400
titanic_like = pd.DataFrame({
    "PassengerId": np.arange(1, n + 1),
    "Pclass": rng.choice([1, 2, 3], n, p=[0.25, 0.2, 0.55]),
    "Sex": rng.choice(["male", "female"], n, p=[0.65, 0.35]),
    "Age": np.where(rng.random(n) < 0.2, np.nan, rng.normal(30, 13, n).clip(1, 80).round()),
    "SibSp": rng.poisson(0.5, n),
    "Fare": rng.gamma(2, 15, n).round(2),
    "Embarked": pd.Series(rng.choice(["C", "Q", "S"], n, p=[0.19, 0.09, 0.72])).mask(rng.random(n) < 0.02),
    "CabinSize": rng.choice(["small", "medium", "large"], n),
})
logit_ = 1.5 * (titanic_like.Sex == "female") - 0.8 * (titanic_like.Pclass - 2) + 0.3 * (titanic_like.Embarked == "C") - 0.4
titanic_like["Survived"] = (rng.random(n) < 1 / (1 + np.exp(-logit_))).astype(int)

display(titanic_like.head())
print(titanic_like.dtypes.to_string())
print("\nMissing values:", titanic_like.isna().sum()[lambda s: s > 0].to_dict())

pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["Age", "SibSp", "Fare"]),
    ("ord", make_pipeline(OrdinalEncoder(categories=[[1, 2, 3], ["small", "medium", "large"]]), StandardScaler()), ["Pclass", "CabinSize"]),
    ("nom", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), ["Sex", "Embarked"]),
])  # PassengerId is dropped automatically (remainder="drop")

model = Pipeline([("pre", pre), ("classifier", KNeighborsClassifier(n_neighbors=15))])
Xt, yt = titanic_like.drop(columns="Survived"), titanic_like["Survived"]
print(f"\n5-fold CV accuracy: {cross_val_score(model, Xt, yt, cv=5).mean():.3f}")
print("Model input columns after preprocessing:", list(model.fit(Xt, yt).named_steps["pre"].get_feature_names_out()))

### ⚠️ Interview traps
- `LabelEncoder` is for the **target** y, not for nominal features.
- `OneHotEncoder(handle_unknown="ignore")` stops a crash when production data contains a new category.
- High-cardinality nominal features (e.g. 5,000 cities) make one-hot vectors huge, which is bad for KNN's curse of dimensionality. Mention grouping rare categories, or target/frequency encoding (done inside CV to avoid leakage).
- IDs look numeric but carry no meaning. Drop them, or KNN will happily measure "distance" between passenger numbers.
- Encode and impute **inside the Pipeline**, so it's fit on the training folds only (Module 1).

### 🏋️ Exercise B.1 – Classify the variable types
Use one of: `'target'`, `'numerical'`, `'nominal'`, `'ordinal'`, `'identifier'`.

In [ ]:
column_types = {
    "PassengerId": None,
    "Pclass": None,
    "Sex": None,
    "Age": None,
    "Fare": None,
    "Embarked": None,
    "CabinSize": None,
    "Survived": None,
}
# TODO: replace every None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
column_types = {
    "PassengerId": "identifier",
    "Pclass": "ordinal",
    "Sex": "nominal",
    "Age": "numerical",
    "Fare": "numerical",
    "Embarked": "nominal",
    "CabinSize": "ordinal",
    "Survived": "target",
}
```

</details>

In [ ]:
_types_key = ["identifier", "ordinal", "nominal", "numerical", "numerical", "nominal", "ordinal", "target"]
check("Ex B.1 column types", lambda: None if None in column_types.values() else list(column_types.values()), _types_key)

### 🏋️ Exercise B.2 – Build the preprocessor yourself
Return a `ColumnTransformer` that: imputes (median) and scales **numerical** columns; ordinal-encodes (with the given order) and scales **ordinal** columns; imputes (most frequent) and one-hot encodes **nominal** columns with `handle_unknown='ignore'`. Identifier columns must be dropped.

In [ ]:
def build_preprocessor(numerical, ordinal, ordinal_orders, nominal):
    """ordinal_orders: list of category lists, one per ordinal column, lowest → highest."""
    # TODO: return ColumnTransformer([("num", ...), ("ord", ...), ("nom", ...)])
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def build_preprocessor(numerical, ordinal, ordinal_orders, nominal):
    """ordinal_orders: list of category lists, one per ordinal column, lowest → highest."""
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numerical),
        ("ord", make_pipeline(OrdinalEncoder(categories=ordinal_orders), StandardScaler()), ordinal),
        ("nom", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), nominal),
    ])
```

</details>

In [ ]:
my_pre = build_preprocessor(["Age", "SibSp", "Fare"], ["Pclass", "CabinSize"], [[1, 2, 3], ["small", "medium", "large"]], ["Sex", "Embarked"])

def _b2_shape():
    if my_pre is None:
        return None
    out = my_pre.fit_transform(Xt)
    return out.shape[1]          # 3 numerical + 2 ordinal + 2 (Sex) + 3 (Embarked) = 10

def _b2_unseen():
    if my_pre is None:
        return None
    new_row = Xt.iloc[[0]].copy(); new_row["Embarked"] = "Z"     # port never seen in training
    return np.isfinite(np.asarray(my_pre.transform(new_row).todense() if hasattr(my_pre.transform(new_row), "todense")
                                  else my_pre.transform(new_row))).all()

check("Ex B.2 output has 10 columns (PassengerId dropped)", _b2_shape, 10)
check("Ex B.2 handles an unseen category", _b2_unseen)

---
<a id="m1"></a>
# 🔴 Module 1 – Data Leakage, Cross-Validation & Time-Series Validation (Q3, Q10)

### 📘 Recap
**Data leakage** = information that would not be available at prediction time influences training or model selection.
It makes scores look better than real-world performance.

| Leak | Example from the course | Fix |
|---|---|---|
| Tuning on the test set | Titanic `best_k = argmax(test accuracy)` | `GridSearchCV` on train; test **once** |
| Preprocessing before split | `StandardScaler().fit(X)` on all rows | Put scalers/imputers/selectors **inside a `Pipeline`** |
| Shuffling time series | Shuffled K-fold on robot telemetry / weather | `TimeSeriesSplit` or chronological holdout |
| Target leakage | A feature computed from the label | Domain review of every feature |

**The three data roles**
- **Train** → fit parameters. **Validation (or CV folds)** → choose hyperparameters/models. **Test** → one final, unbiased estimate.

**K-fold CV:** split training data into *k* folds → train *k* models, each validated on a different fold →
report **mean ± std** of the *k* scores → refit the chosen configuration on all training data.

### 🧪 Demo 1a – Feature selection *before* CV leaks (pure-noise data!)
The labels below are random coin flips, so honest accuracy should be ≈ 50%. Watch what leaking does.

In [ ]:
rng = np.random.default_rng(RNG)
X_noise = rng.normal(size=(100, 5000))      # 5000 random features
y_noise = rng.integers(0, 2, size=100)      # random labels → nothing to learn

# ❌ LEAKY: select the 20 "best" features using ALL rows, then cross-validate
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(KNeighborsClassifier(), X_selected, y_noise, cv=5)

# ✅ CORRECT: selection happens inside each training fold
honest_pipe = make_pipeline(SelectKBest(f_classif, k=20), KNeighborsClassifier())
honest = cross_val_score(honest_pipe, X_noise, y_noise, cv=5)

print(f"Leaky CV accuracy : {leaky.mean():.2f} ± {leaky.std():.2f}   ← looks like a great model on random data!")
print(f"Honest CV accuracy: {honest.mean():.2f} ± {honest.std():.2f}   ← the truth (≈ coin flip)")

### 🧪 Demo 1b – The correct way to tune `k` (fixing your Titanic loop)
We use `load_breast_cancer` (bundled, offline) in place of Titanic. The pattern is identical.

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RNG)

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),          # re-fit inside every CV training fold
    ("classifier", KNeighborsClassifier()),
])

grid = GridSearchCV(knn_pipeline,
                    param_grid={"classifier__n_neighbors": list(range(1, 26))},
                    cv=StratifiedKFold(5, shuffle=True, random_state=RNG),
                    scoring="accuracy")
grid.fit(X_train, y_train)                 # the test set is never touched here

cv_means = grid.cv_results_["mean_test_score"]
cv_stds = grid.cv_results_["std_test_score"]
ks = grid.cv_results_["param_classifier__n_neighbors"].data.astype(int)

plt.errorbar(ks, cv_means, yerr=cv_stds, marker="o", capsize=3)
plt.xlabel("k (n_neighbors)"); plt.ylabel("5-fold CV accuracy (mean ± std)")
plt.title("Choosing k with cross-validation on TRAINING data only")
plt.show()

print("Best k (by CV):", grid.best_params_["classifier__n_neighbors"])
print(f"CV accuracy    : {grid.best_score_:.3f}")
print(f"Test accuracy  : {grid.score(X_test, y_test):.3f}   ← reported once, at the very end")

### 🧪 Demo 1c – Why shuffled K-fold lies on time series
A trending signal (like slowly rising robot current or exchange rates). KNN regression on the time index.

In [ ]:
t = np.arange(600).reshape(-1, 1)
signal = 0.05 * t.ravel() + np.sin(t.ravel() / 15) * 3 + np.random.default_rng(RNG).normal(0, 0.8, 600)

model = KNeighborsRegressor(n_neighbors=5)
shuffled = cross_val_score(model, t, signal, cv=KFold(5, shuffle=True, random_state=RNG), scoring="r2")
forward = cross_val_score(model, t, signal, cv=TimeSeriesSplit(n_splits=5), scoring="r2")

print(f"Shuffled K-fold R²  : {shuffled.mean():.3f}  ← 'future' neighbours leak into training")
print(f"TimeSeriesSplit R²  : {forward.mean():.3f}  ← honest: KNN can't extrapolate into the future")

# Visualize the TimeSeriesSplit folds
fig, ax = plt.subplots(figsize=(9, 2.8))
for i, (tr, va) in enumerate(TimeSeriesSplit(n_splits=5).split(t)):
    ax.scatter(tr, [i] * len(tr), c="tab:blue", s=2, label="train" if i == 0 else None)
    ax.scatter(va, [i] * len(va), c="tab:orange", s=2, label="validate" if i == 0 else None)
ax.set_yticks(range(5)); ax.set_yticklabels([f"fold {i+1}" for i in range(5)])
ax.set_xlabel("time index"); ax.set_title("TimeSeriesSplit: always train on the past, validate on the future")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

### ⚠️ Interview traps
- *"I picked the best k using the test set."* → Instant red flag. Say **"validation / cross-validation"**, never "test", when tuning.
- *"I scaled the whole dataset first, then split."* → Leakage. Scalers are **fit on train, applied to test**.
- *"CV score = 0.95."* → Always add **± std**, and mention that the final model is **refit on all training data**.
- Time series → **never shuffle**. Say "forward-chaining / `TimeSeriesSplit`".

### 🏋️ Exercise 1.1 – Rewrite the leaky loop
Complete `tune_k_properly` so it returns `(best_k, cv_score, test_score)` using `GridSearchCV` with 5-fold CV on the **training** data only.

In [ ]:
def tune_k_properly(X_train, y_train, X_test, y_test, k_values=range(1, 26)):
    """Return (best_k, best_cv_score, test_score)."""
    # TODO 1: build a Pipeline with StandardScaler and KNeighborsClassifier (step name "classifier")
    # TODO 2: GridSearchCV over classifier__n_neighbors with cv=5, scoring="accuracy"
    # TODO 3: fit on the TRAINING data only
    # TODO 4: return best k, best CV score, and test score (evaluate the test set exactly once)
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def tune_k_properly(X_train, y_train, X_test, y_test, k_values=range(1, 26)):
    """Return (best_k, best_cv_score, test_score)."""
    pipe = Pipeline([("scaler", StandardScaler()), ("classifier", KNeighborsClassifier())])
    gs = GridSearchCV(pipe, {"classifier__n_neighbors": list(k_values)}, cv=5, scoring="accuracy")
    gs.fit(X_train, y_train)
    return gs.best_params_["classifier__n_neighbors"], gs.best_score_, gs.score(X_test, y_test)
```

</details>

In [ ]:
result = tune_k_properly(X_train, y_train, X_test, y_test)
if result is not None:
    print("best_k=%d  cv=%.3f  test=%.3f" % result)
check("Ex 1.1 returns sensible (k, cv, test)",
      lambda: None if result is None else (1 <= result[0] <= 25 and 0.85 < result[1] <= 1 and 0.85 < result[2] <= 1))

### 🏋️ Exercise 1.2 – Report CV like a professional
Return the **mean and std** of 5-fold R² for a scaled `KNeighborsRegressor(n_neighbors=10)` on the diabetes dataset.

In [ ]:
def cv_report(X, y):
    """Return (mean_r2, std_r2) from 5-fold CV of make_pipeline(StandardScaler(), KNeighborsRegressor(10))."""
    # TODO: use cross_val_score(..., cv=KFold(5, shuffle=True, random_state=RNG), scoring="r2")
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def cv_report(X, y):
    """Return (mean_r2, std_r2) from 5-fold CV of make_pipeline(StandardScaler(), KNeighborsRegressor(10))."""
    scores = cross_val_score(make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10)), X, y,
                             cv=KFold(5, shuffle=True, random_state=RNG), scoring="r2")
    return scores.mean(), scores.std()
```

</details>

In [ ]:
Xd, yd = load_diabetes(return_X_y=True)
rep = cv_report(Xd, yd)
if rep is not None:
    print(f"KNN regression R² = {rep[0]:.3f} ± {rep[1]:.3f}")
check("Ex 1.2 mean R² in a plausible range", lambda: None if rep is None else (0.3 < rep[0] < 0.55 and rep[1] > 0))

### 🏋️ Exercise 1.3 – Spot the leak (written)
For each snippet, write **leak / no leak** and one sentence why.

```python
# (a)
X_scaled = StandardScaler().fit_transform(X); X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y)
# (b)
pipe = make_pipeline(SimpleImputer(), StandardScaler(), KNeighborsClassifier()); cross_val_score(pipe, X_tr, y_tr, cv=5)
# (c)
for depth in range(1, 15): tree.set_params(max_depth=depth).fit(X_tr, y_tr); scores.append(tree.score(X_te, y_te))
# (d)
cross_val_score(model, robot_df[features], robot_df['current'], cv=KFold(5, shuffle=True))
```

> ✍️ *Your answers here*

<details><summary><b>💡 Show solution</b></summary>

(a) **Leak** – scaler saw test rows' mean/std.  (b) **No leak** – preprocessing is refit per fold.  (c) **Leak** – hyperparameter chosen on the test set.  (d) **Leak** – shuffled K-fold on time-series data trains on the future.
</details>

---
<a id="m2"></a>
# 🔴 Module 2 – Bias–Variance Trade-off & Model Complexity (Q6)

### 📘 Recap
$$\text{Expected error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible noise}$$

| | Underfitting (high **bias**) | Sweet spot | Overfitting (high **variance**) |
|---|---|---|---|
| Training error | high | low-ish | very low |
| Validation error | high | **lowest** | high |
| Train–val gap | small | small | **large** |
| Fix | more complexity / features | — | less complexity, regularize, more data |

**Complexity knobs you used in this course**

| Model | Knob | → Overfit | → Underfit |
|---|---|---|---|
| Polynomial regression | `degree` | high degree | degree 0–1 |
| KNN | `n_neighbors` (k) | **small k** (k=1) | **large k** |
| Decision tree | `max_depth`, `min_samples_leaf` | deep, tiny leaves | shallow, big leaves |
| Logistic regression | `C` (inverse regularization) | large C | small C |

### 🧪 Demo 2a – Lab 2 revisited: polynomial degree 0–5 (and beyond) on BMI → diabetes progression

In [ ]:
diab = load_diabetes(as_frame=True)
X_bmi = diab.data[["bmi"]].values
y_prog = diab.target.values
cv = KFold(5, shuffle=True, random_state=RNG)

def poly_model(degree):
    if degree == 0:
        return DummyRegressor(strategy="mean")              # degree 0 = predict the mean
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), LinearRegression())

degrees = list(range(0, 13))
train_mse, val_mse = [], []
for d in degrees:
    res = cross_validate(poly_model(d), X_bmi, y_prog, cv=cv, scoring="neg_mean_squared_error", return_train_score=True)
    train_mse.append(-res["train_score"].mean())
    val_mse.append(-res["test_score"].mean())

plt.plot(degrees, train_mse, "o-", label="training MSE")
plt.plot(degrees, val_mse, "s-", label="validation MSE (5-fold CV)")
plt.axvspan(-0.3, 0.5, color="red", alpha=0.08, label="underfit")
plt.axvspan(7.5, 12.3, color="purple", alpha=0.08, label="overfit")
plt.yscale("log")                                         # high degrees explode, so use a log axis
plt.xlabel("polynomial degree"); plt.ylabel("MSE (log scale)"); plt.legend()
plt.title("Training error keeps falling; validation error is U-shaped")
plt.show()

display(pd.DataFrame({"degree": degrees, "train_MSE": np.round(train_mse, 1), "val_MSE": np.round(val_mse, 1)}).head(6))
print("Best degree by CV:", degrees[int(np.argmin(val_mse))])

### 🧪 Demo 2b – The same picture for KNN (`k`) and decision trees (`max_depth`)

In [ ]:
X_all, y_all = load_diabetes(return_X_y=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

k_range = np.arange(1, 41)
tr, va = validation_curve(make_pipeline(StandardScaler(), KNeighborsRegressor()), X_all, y_all,
                          param_name="kneighborsregressor__n_neighbors", param_range=k_range, cv=cv, scoring="r2")
axes[0].plot(k_range, tr.mean(1), label="train R²"); axes[0].plot(k_range, va.mean(1), label="validation R²")
axes[0].set_xlabel("k  (← overfit | underfit →)"); axes[0].set_title("KNN: small k = high variance"); axes[0].legend()

depths = np.arange(1, 16)
tr, va = validation_curve(DecisionTreeRegressor(random_state=RNG), X_all, y_all,
                          param_name="max_depth", param_range=depths, cv=cv, scoring="r2")
axes[1].plot(depths, tr.mean(1), label="train R²"); axes[1].plot(depths, va.mean(1), label="validation R²")
axes[1].set_xlabel("max_depth  (← underfit | overfit →)"); axes[1].set_title("Tree: deep = high variance"); axes[1].legend()
plt.tight_layout(); plt.show()

### ⚠️ Interview traps
- Choosing complexity by **training** error always picks the most complex model. Always use **validation/CV** error.
- KNN's knob runs **backwards** compared with trees and polynomials: *smaller* k = *more* complex.
- "How do you know you're overfitting?" → **large gap** between training and validation scores.

### 🏋️ Exercise 2.1 – Diagnose a model
Complete `diagnose(train_score, val_score)` returning `'underfit'`, `'overfit'` or `'good'` for R² scores. Rules: if val < 0.3 and train < 0.4 → underfit; elif train − val > 0.15 → overfit; else good.

In [ ]:
def diagnose(train_score, val_score):
    # TODO: implement the rules above
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def diagnose(train_score, val_score):
    if val_score < 0.3 and train_score < 0.4:
        return "underfit"
    if train_score - val_score > 0.15:
        return "overfit"
    return "good"
```

</details>

In [ ]:
check("Ex 2.1 underfit", lambda: diagnose(0.10, 0.08), "underfit")
check("Ex 2.1 overfit", lambda: diagnose(1.00, 0.20), "overfit")
check("Ex 2.1 good", lambda: diagnose(0.55, 0.48), "good")

### 🏋️ Exercise 2.2 – Find the best tree depth
Use `validation_curve` (5-fold, R²) for `DecisionTreeRegressor(random_state=RNG)` on the full diabetes data with depths 1–15. Return the depth with the highest mean validation R² and use `diagnose` on depth 15.

In [ ]:
def best_tree_depth(X, y):
    depths = np.arange(1, 16)
    # TODO: compute train/validation scores with validation_curve, return (best_depth, diagnose(train@15, val@15))
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def best_tree_depth(X, y):
    depths = np.arange(1, 16)
    tr, va = validation_curve(DecisionTreeRegressor(random_state=RNG), X, y, param_name="max_depth",
                              param_range=depths, cv=KFold(5, shuffle=True, random_state=RNG), scoring="r2")
    best = int(depths[np.argmax(va.mean(axis=1))])
    return best, diagnose(tr.mean(axis=1)[-1], va.mean(axis=1)[-1])
```

</details>

In [ ]:
res = best_tree_depth(X_all, y_all)
print(res)
check("Ex 2.2 best depth is shallow (≤ 5)", lambda: None if res is None else res[0] <= 5)
check("Ex 2.2 depth 15 is overfit", lambda: None if res is None else res[1], "overfit")

---
<a id="m3"></a>
# 🔴 Module 3 – Classification Metrics, Imbalance & Thresholds (Q9)

### 📘 Recap
|  | **Predicted −** | **Predicted +** |
|---|---|---|
| **Actual −** | TN | FP (false alarm, Type I) |
| **Actual +** | FN (miss, Type II) | TP |

$$\text{Precision}=\frac{TP}{TP+FP}\qquad \text{Recall}=\frac{TP}{TP+FN}\qquad F_1=\frac{2PR}{P+R}\qquad \text{Accuracy}=\frac{TP+TN}{N}$$

- 🧠 **P**recision → the **P**redicted-positive column. **R**ecall → the **R**eal-positive row.
- **Imbalanced data:** accuracy is misleading. Always compare with a `DummyClassifier`.
- **Threshold:** lowering it on `predict_proba` → recall ↑, precision ↓ (and vice versa).
- **Which metric?** Cost of a FN high (disease, fraud, robot failure) → **recall**. Cost of a FP high (spam, legal flagging) → **precision**.

### 🧪 Demo 3a – The accuracy paradox: “Is this digit a 5?”
`load_digits` is a small, bundled version of MNIST (8×8 images), so the same workshop experiment runs offline.

In [ ]:
digits = load_digits()
X_dig, y_is5 = digits.data, (digits.target == 5).astype(int)
print(f"Share of 5s: {y_is5.mean():.1%}")

Xtr, Xte, ytr, yte = train_test_split(X_dig, y_is5, test_size=0.25, stratify=y_is5, random_state=RNG)
dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)

for name, m in [("Dummy (always 'not 5')", dummy), ("Logistic regression", logreg)]:
    p = m.predict(Xte)
    print(f"{name:24s} acc={accuracy_score(yte, p):.3f}  precision={precision_score(yte, p, zero_division=0):.3f}"
          f"  recall={recall_score(yte, p):.3f}  f1={f1_score(yte, p):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, m) in zip(axes, [("Dummy", dummy), ("LogReg", logreg)]):
    ConfusionMatrixDisplay.from_estimator(m, Xte, yte, display_labels=["not 5", "5"], ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

### 🧪 Demo 3b – Screening: trading precision for recall with the threshold
`load_breast_cancer` stands in for UCI Heart Disease (bundled, offline). **Positive = malignant**, the case we must not miss.

In [ ]:
bc = load_breast_cancer()
X_bc, y_bc = bc.data, (bc.target == 0).astype(int)      # 1 = malignant
Xtr, Xte, ytr, yte = train_test_split(X_bc, y_bc, test_size=0.25, stratify=y_bc, random_state=RNG)

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=0.1))
# out-of-fold probabilities on TRAIN → choose threshold without touching the test set (Module 1!)
proba_oof = cross_val_predict(clf, Xtr, ytr, cv=5, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(ytr, proba_oof)

plt.plot(thr, prec[:-1], label="precision"); plt.plot(thr, rec[:-1], label="recall")
plt.axvline(0.5, ls="--", c="grey", label="default 0.5")
plt.xlabel("decision threshold"); plt.legend(); plt.title("Lower threshold → higher recall, lower precision")
plt.show()

clf.fit(Xtr, ytr)
p_test = clf.predict_proba(Xte)[:, 1]
for t in [0.5, 0.3, 0.1]:
    pred = (p_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    print(f"threshold={t:.1f}  recall={tp/(tp+fn):.3f}  precision={tp/(tp+fp):.3f}  missed cancers (FN)={fn}  false alarms (FP)={fp}")

### ⚠️ Interview traps
- "My model has 90% accuracy" on imbalanced data → interviewer asks *"what does a dummy get?"*
- Mixing up precision and recall. Practise: *"Of the patients I flagged, how many were sick?"* = precision.
- Choosing the threshold on the test set is leakage too. Use out-of-fold (CV) probabilities on train.
- `classification_report` shows metrics **per class**; know macro vs weighted averages for multi-class.

### 🏋️ Exercise 3.1 – Metrics by hand
Implement the four metrics from raw counts (do **not** use sklearn inside).

In [ ]:
def metrics_by_hand(y_true, y_pred):
    """Return (accuracy, precision, recall, f1) for binary labels where 1 is positive."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    # TODO: compute TP, FP, FN, TN with boolean masks, then the four metrics
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def metrics_by_hand(y_true, y_pred):
    """Return (accuracy, precision, recall, f1) for binary labels where 1 is positive."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return acc, prec, rec, f1
```

</details>

In [ ]:
yt = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
yp = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]     # TP=2, FN=2, FP=1, TN=5
expected = (accuracy_score(yt, yp), precision_score(yt, yp), recall_score(yt, yp), f1_score(yt, yp))
check("Ex 3.1 metrics_by_hand", lambda: metrics_by_hand(yt, yp), expected)

### 🏋️ Exercise 3.2 – Pick a screening threshold
Using the out-of-fold probabilities `proba_oof` from Demo 3b, return the **highest** threshold whose recall on train is ≥ 0.98.

In [ ]:
def threshold_for_recall(y_true, proba, target_recall=0.98):
    # TODO: use precision_recall_curve; keep thresholds where recall >= target; return the max of those
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def threshold_for_recall(y_true, proba, target_recall=0.98):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    ok = thr[rec[:-1] >= target_recall]
    return float(ok.max())
```

</details>

In [ ]:
t_star = threshold_for_recall(ytr, proba_oof)
check("Ex 3.2 threshold is below the default 0.5", lambda: None if t_star is None else t_star < 0.5)
if t_star is not None:
    pred = (p_test >= t_star).astype(int)
    print(f"threshold={t_star:.3f}  test recall={recall_score(yte, pred):.3f}  test precision={precision_score(yte, pred):.3f}")

---
<a id="m4"></a>
# 🟠 Module 4 – Gradient Descent & Learning Rate (Q4)

### 📘 Recap
Model $\hat y = \theta_0 + \theta_1 x$ and cost (MSE)
$$J(\theta_0,\theta_1)=\frac{1}{2m}\sum_{i=1}^{m}(\hat y_i-y_i)^2$$
Partial derivatives (chain rule: $\partial\hat y/\partial\theta_0 = 1$, $\partial\hat y/\partial\theta_1 = x$):
$$\frac{\partial J}{\partial\theta_0}=\frac{1}{m}\sum(\hat y_i-y_i)\qquad \frac{\partial J}{\partial\theta_1}=\frac{1}{m}\sum(\hat y_i-y_i)\,x_i$$
Update: $\theta \leftarrow \theta - \alpha\,\nabla J$.

| Learning rate α | Loss curve looks like | Cause |
|---|---|---|
| too small | slow, steady decline; not converged at the end | tiny steps |
| good | fast drop then flat | — |
| too large | zig-zag, then **explodes** to inf / nan | steps overshoot the minimum |

> ❗ **Correcting your Q4 answer:** a large learning rate is an *optimization* problem (divergence), **not** overfitting.
> Overfitting is about *model capacity* versus the amount of data (Module 2).

| Variant | Rows per update | Pros | Cons |
|---|---|---|---|
| Batch GD | all *m* | exact, smooth gradient | slow on big data |
| SGD (`SGDRegressor`) | 1 | fast, online learning | noisy; needs LR decay |
| Mini-batch | 32–256 | vectorized + stable | batch size is another hyperparameter |

### 🧪 Demo 4a – Three learning rates on the same data

In [ ]:
rng = np.random.default_rng(RNG)
x_gd = rng.uniform(0, 2, 200)
y_gd = 4 + 3 * x_gd + rng.normal(0, 0.5, 200)

def batch_gd(x, y, lr, epochs=200):
    """Same update rule as src/model.py in the Linear Regression Architecture workshop, plus a loss history."""
    m = len(x); t0 = t1 = 0.0; history = []
    for _ in range(epochs):
        err = (t0 + t1 * x) - y
        history.append((err ** 2).mean() / 2)
        t0 -= lr * err.mean()
        t1 -= lr * (err * x).mean()
    return t0, t1, history

with np.errstate(over="ignore", invalid="ignore"):
    runs = {lr: batch_gd(x_gd, y_gd, lr) for lr in [0.01, 0.3, 1.2]}

fig, ax = plt.subplots()
for lr, (t0, t1, h) in runs.items():
    final = f"θ0={t0:.2f}, θ1={t1:.2f}" if np.isfinite(t1) and abs(t1) < 1e3 else "DIVERGED"
    ax.plot(np.clip(np.nan_to_num(h, nan=1e6, posinf=1e6), 1e-3, 1e6), label=f"lr={lr}  → {final}")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("cost J (log scale)"); ax.legend()
ax.set_title("Too small (slow) · good · too large (diverges)")
plt.show()

sk = LinearRegression().fit(x_gd.reshape(-1, 1), y_gd)
print(f"sklearn closed form: θ0={sk.intercept_:.2f}, θ1={sk.coef_[0]:.2f}  (lr=0.3 should match)")

### ⚠️ Interview traps
- Be able to **derive** ∂J/∂θ₁ on a whiteboard: the `* x` comes from the chain rule.
- Unscaled features make the cost surface a long narrow valley, so gradient descent zig-zags. **Scale before GD.**
- `nan` in your weights almost always means the learning rate is too large (or the features are unscaled).
- An **epoch** is one full pass over the data. The number of updates per epoch is 1 (batch), m (SGD) or m / batch_size (mini-batch).

### 🏋️ Exercise 4.1 – Implement mini-batch gradient descent
Shuffle each epoch, iterate over batches of `batch_size`, and apply the same update using only that batch. Return `(theta0, theta1)`.

In [ ]:
def minibatch_gd(x, y, lr=0.1, epochs=100, batch_size=32, seed=RNG):
    rng = np.random.default_rng(seed)
    t0 = t1 = 0.0
    # TODO: for each epoch: permute indices; for each batch slice: compute err on the batch and update t0, t1
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def minibatch_gd(x, y, lr=0.1, epochs=100, batch_size=32, seed=RNG):
    rng = np.random.default_rng(seed)
    t0 = t1 = 0.0
    for _ in range(epochs):
        idx = rng.permutation(len(x))
        for start in range(0, len(x), batch_size):
            b = idx[start:start + batch_size]
            err = (t0 + t1 * x[b]) - y[b]
            t0 -= lr * err.mean()
            t1 -= lr * (err * x[b]).mean()
    return t0, t1
```

</details>

In [ ]:
check("Ex 4.1 mini-batch matches closed form (±0.1)",
      lambda: minibatch_gd(x_gd, y_gd), (sk.intercept_, sk.coef_[0]), atol=0.1)

### 🏋️ Exercise 4.2 – Explain it (written)
In 3–4 sentences: why does `lr=1.2` diverge on this data while `lr=0.3` converges? What would you check first if `SGDRegressor` returned huge coefficients?

> ✍️ *Your answer here*

---
<a id="m5"></a>
# 🟠 Module 5 – Regression Metrics: R², MAE, RMSE, MAPE (Q5)

### 📘 Recap
| Metric | Formula | Units | Read it as |
|---|---|---|---|
| MAE | $\frac1n\sum\lvert y-\hat y\rvert$ | target units | typical error size |
| RMSE | $\sqrt{\frac1n\sum(y-\hat y)^2}$ | target units | error with **big misses penalized** |
| R² | $1-\frac{SS_{res}}{SS_{tot}} = 1-\frac{\sum(y-\hat y)^2}{\sum(y-\bar y)^2}$ | unitless | **share of variance explained** vs predicting the mean |
| MAPE | $\frac{100}{n}\sum\left\lvert\frac{y-\hat y}{y}\right\rvert$ | % | relative error, easy for stakeholders |

- RMSE ≥ MAE **always**. A big gap means a few large errors (outliers).
- R² can be **negative** on test data, meaning the model is worse than predicting the mean.
- MAPE breaks when y = 0, explodes when y is near 0, and is asymmetric.

**Your California result explained:** R² = 0.46 → MedInc alone explains ~46% of the variance in house value.
MAE 0.63 → off by ≈ **$63k** on average (target in $100k). RMSE 0.84 > MAE → some large misses (the $500k cap).

### 🧪 Demo 5 – Metrics by hand, and what outliers do

In [ ]:
y_true = np.array([200, 250, 300, 350, 400], dtype=float)
y_pred_even = np.array([220, 230, 320, 330, 420], dtype=float)   # every error = 20
y_pred_outl = np.array([200, 250, 300, 350, 300], dtype=float)   # one error = 100

def report(y, p):
    return dict(MAE=mean_absolute_error(y, p), RMSE=np.sqrt(mean_squared_error(y, p)),
                R2=r2_score(y, p), MAPE=mean_absolute_percentage_error(y, p) * 100)

display(pd.DataFrame({"even errors": report(y_true, y_pred_even), "one big error": report(y_true, y_pred_outl)}).round(2))

print("Same total absolute error (100) — but RMSE punishes the single big miss much more.")
print("Negative R² example:", round(r2_score(y_true, np.full(5, 500.0)), 2), "(worse than predicting the mean)")
print("MAPE near zero:", round(mean_absolute_percentage_error([0.01, 100], [1, 100]) * 100, 1), "%  ← one tiny y dominates")

### ⚠️ Interview traps
- Don't say R² "measures variance". Say it is the **proportion of variance in y explained by the model**.
- Always translate errors into **business units** ("about $63k off").
- Report several metrics. R² alone hides the size of the errors, and MAE alone hides the big misses.

### 🏋️ Exercise 5.1 – Implement R², RMSE and MAPE with NumPy only

In [ ]:
def r2_manual(y, p):
    # TODO: 1 - SS_res / SS_tot
    return None

def rmse_manual(y, p):
    # TODO
    return None

def mape_manual(y, p):
    # TODO: return a percentage; raise ValueError if any y == 0
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def r2_manual(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)

def rmse_manual(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return np.sqrt(np.mean((y - p) ** 2))

def mape_manual(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    if np.any(y == 0):
        raise ValueError("MAPE undefined when y contains 0")
    return np.mean(np.abs((y - p) / y)) * 100
```

</details>

In [ ]:
check("Ex 5.1 R²", lambda: r2_manual(y_true, y_pred_outl), r2_score(y_true, y_pred_outl))
check("Ex 5.1 RMSE", lambda: rmse_manual(y_true, y_pred_outl), np.sqrt(mean_squared_error(y_true, y_pred_outl)))
check("Ex 5.1 MAPE", lambda: mape_manual(y_true, y_pred_even), mean_absolute_percentage_error(y_true, y_pred_even) * 100)

def _mape_zero_raises():
    if mape_manual([1, 2], [1, 2]) is None:
        return None
    try:
        mape_manual([0, 2], [1, 2]); return False
    except ValueError:
        return True
check("Ex 5.1 MAPE raises on y = 0", _mape_zero_raises)

### 🏋️ Exercise 5.2 – Stakeholder sentence (written)
A multivariate diabetes model gives R² = 0.48, MAE = 43, MAPE = 38%. The target ranges 25–346. Write two sentences for a clinic manager: what the model can and can't be used for.

> ✍️ *Your answer here*

---
<a id="m6"></a>
# 🟠 Module 6 – Decision Tree Splits by Hand (Q7)

### 📘 Recap – the greedy split algorithm
1. For each feature, sort the unique values. The candidate thresholds are the **midpoints** between neighbours.
2. For each threshold, split into Left (x ≤ t) and Right (x > t).
3. Score the split by the **sample-weighted impurity of the children**:
$$\text{score}=\frac{n_L}{n}\,I(L)+\frac{n_R}{n}\,I(R)$$
   - Regression: $I$ = MSE around the child's mean. The leaf predicts the **mean**.
   - Classification: $I$ = Gini $=1-\sum_k p_k^2$ (or entropy). The leaf predicts the **majority class**.
4. Pick the (feature, threshold) with the lowest score, i.e. the largest impurity reduction. Recurse until a stopping rule is met (`max_depth`, `min_samples_split`, `min_samples_leaf`).

**Q7 worked:** parent MSE = 101 → split at 2.5 gives weighted MSE = 1.0 → leaves predict 11 and 31.

### 🧪 Demo 6 – Verify the Q7 answer with scikit-learn

In [ ]:
x_q7 = np.array([1, 2, 3, 4], dtype=float)
y_q7 = np.array([10, 12, 30, 32], dtype=float)

stump = DecisionTreeRegressor(max_depth=1).fit(x_q7.reshape(-1, 1), y_q7)
print("sklearn threshold :", stump.tree_.threshold[0])
print("leaf predictions  :", stump.predict([[1.5], [3.5]]))

grid_x = np.linspace(0.5, 4.5, 200).reshape(-1, 1)
plt.scatter(x_q7, y_q7, s=80, zorder=3, label="data")
plt.step(grid_x.ravel(), stump.predict(grid_x), where="post", c="tab:red", label="depth-1 tree (piecewise constant)")
plt.axvline(2.5, ls="--", c="grey"); plt.legend(); plt.title("Regression tree = step function")
plt.show()

### ⚠️ Interview traps
- Forgetting to **weight** the child impurities by sample count.
- Saying a regression leaf predicts the median or a regression line. It predicts the **mean** of the training targets in that leaf.
- Trees are **not** sensitive to feature scaling (they only compare thresholds), unlike KNN and gradient descent.

### 🏋️ Exercise 6.1 – Implement `best_split_regression`
Return `(best_threshold, best_weighted_mse)` for one feature.

In [ ]:
def best_split_regression(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    values = np.unique(x)
    candidates = (values[:-1] + values[1:]) / 2      # midpoints
    best_t, best_score = None, np.inf
    for t in candidates:
        left, right = y[x <= t], y[x > t]
        # TODO: compute the weighted MSE of left/right around their own means; keep the minimum
        pass
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def best_split_regression(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    values = np.unique(x)
    candidates = (values[:-1] + values[1:]) / 2      # midpoints
    best_t, best_score = None, np.inf
    for t in candidates:
        left, right = y[x <= t], y[x > t]
        score = (len(left) * left.var() + len(right) * right.var()) / len(y)   # var() = MSE around the mean
        if score < best_score:
            best_t, best_score = t, score
    return best_t, best_score
```

</details>

In [ ]:
check("Ex 6.1 Q7 data", lambda: best_split_regression(x_q7, y_q7), (2.5, 1.0))
xr = np.random.default_rng(1).uniform(0, 10, 50); yr = np.where(xr > 6.3, 5.0, 1.0) + np.random.default_rng(2).normal(0, .1, 50)
skt = DecisionTreeRegressor(max_depth=1).fit(xr.reshape(-1, 1), yr).tree_.threshold[0]
check("Ex 6.1 matches sklearn on random data", lambda: None if best_split_regression(xr, yr) is None else best_split_regression(xr, yr)[0], skt)

### 🏋️ Exercise 6.2 – Gini for a classification split
Implement `gini(labels)` and `weighted_gini(left, right)`.

In [ ]:
def gini(labels):
    # TODO: 1 - sum(p_k^2)
    return None

def weighted_gini(left, right):
    # TODO: sample-weighted average of gini(left) and gini(right)
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def gini(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return 1 - np.sum(p ** 2)

def weighted_gini(left, right):
    n = len(left) + len(right)
    return len(left) / n * gini(left) + len(right) / n * gini(right)
```

</details>

In [ ]:
check("Ex 6.2 gini pure", lambda: gini([1, 1, 1]), 0.0)
check("Ex 6.2 gini 50/50", lambda: gini([0, 1, 0, 1]), 0.5)
check("Ex 6.2 weighted gini", lambda: weighted_gini([0, 0, 0, 1], [1, 1]), 4/6 * 0.375)

---
<a id="m7"></a>
# 🟡 Module 7 – Logistic Regression from Scratch (Q8 refresher)

### 📘 Recap (memorize these three lines)
$$z=\mathbf{w}^\top\mathbf{x}+b\qquad p=\sigma(z)=\frac{1}{1+e^{-z}}\qquad L=-\frac1m\sum\big[y\log p+(1-y)\log(1-p)\big]$$
Gradient: $\nabla_w L=\frac1m X^\top(p-y)$, $\;\partial L/\partial b=\text{mean}(p-y)$. It has the same shape as the linear-regression gradient.

- Predict 1 if $p \ge 0.5 \iff z \ge 0$, so the **decision boundary is linear**.
- BCE is **convex** with the sigmoid, so gradient descent finds the global minimum. You got this right in the interview ✔

### 🧪 Demo 7 – Workshop data: hours studied → pass/fail

In [ ]:
hours = np.array([0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75, 3, 3.25, 3.5, 3.75, 4, 4.25, 4.5, 4.75, 5])
passed = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1])

lin = LinearRegression().fit(hours.reshape(-1, 1), passed)
logit = LogisticRegression(C=1e6).fit(hours.reshape(-1, 1), passed)   # large C ≈ no regularization

grid_h = np.linspace(-1, 7, 200).reshape(-1, 1)
plt.scatter(hours, passed, c="k", zorder=3, label="data")
plt.plot(grid_h, lin.predict(grid_h), label="linear regression (goes below 0 / above 1!)")
plt.plot(grid_h, logit.predict_proba(grid_h)[:, 1], label="logistic regression P(pass)")
plt.axhline(0.5, ls=":", c="grey")
boundary = -logit.intercept_[0] / logit.coef_[0, 0]
plt.axvline(boundary, ls="--", c="tab:green", label=f"decision boundary ≈ {boundary:.2f} h")
plt.ylim(-0.4, 1.4); plt.xlabel("hours studied"); plt.legend(); plt.show()

### 🏋️ Exercise 7.1 – Sigmoid, BCE and a training loop

In [ ]:
def sigmoid(z):
    # TODO
    return None

def bce(y, p, eps=1e-12):
    # TODO: clip p to [eps, 1-eps] to avoid log(0)
    return None

def train_logistic(x, y, lr=0.1, epochs=20000):
    w = b = 0.0
    # TODO: loop: p = sigmoid(w*x + b); w -= lr*mean((p-y)*x); b -= lr*mean(p-y)
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce(y, p, eps=1e-12):
    p = np.clip(np.asarray(p, float), eps, 1 - eps)
    y = np.asarray(y, float)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def train_logistic(x, y, lr=0.1, epochs=20000):
    w = b = 0.0
    for _ in range(epochs):
        p = sigmoid(w * x + b)
        w -= lr * np.mean((p - y) * x)
        b -= lr * np.mean(p - y)
    return w, b
```

</details>

In [ ]:
check("Ex 7.1 sigmoid(0)", lambda: sigmoid(0), 0.5)
check("Ex 7.1 bce", lambda: bce([1, 0], [0.9, 0.1]), -np.log(0.9))
wb = train_logistic(hours, passed)
check("Ex 7.1 boundary matches sklearn (±0.1 h)",
      lambda: None if wb is None else -wb[1] / wb[0], boundary, atol=0.1)

---
<a id="m8"></a>
# 🟡 Module 8 – KNN Internals, Scaling & the Curse of Dimensionality (Q1, Q2, Q11 refresher)

### 📘 Recap
- **Parametric** (linear and logistic regression): a fixed number of parameters. **Non-parametric** (KNN, trees): complexity grows with the data.
- `KNeighborsClassifier.fit()` **stores** the training data (and optionally builds a KD/Ball tree). `predict()` computes distances → O(n·d) per query.
- **Min-Max to [a, b]:** $x' = a + \frac{(x-x_{min})(b-a)}{x_{max}-x_{min}}$ · **Standardize:** $z=\frac{x-\mu}{\sigma}$ · **Robust:** $\frac{x-\text{median}}{IQR}$
- **Curse of dimensionality:** as d grows, points become sparse and distances concentrate (nearest ≈ farthest).
- **Avoid KNN when:** n is large or low latency is required · data is high-dimensional or sparse · interpretability is required · you need to extrapolate.

### 🧪 Demo 8a – `fit` is instant, `predict` does the work (lazy learning)

In [ ]:
rng = np.random.default_rng(RNG)
Xbig, ybig = rng.normal(size=(50_000, 20)), rng.integers(0, 2, 50_000)
knn = KNeighborsClassifier(n_neighbors=5, algorithm="brute")

t0 = time.perf_counter(); knn.fit(Xbig, ybig); t_fit = time.perf_counter() - t0
t0 = time.perf_counter(); knn.predict(Xbig[:2000]); t_pred = time.perf_counter() - t0
print(f"fit: {t_fit*1000:.1f} ms   predict 2,000 rows: {t_pred*1000:.0f} ms")
print("Stored training samples:", knn.n_samples_fit_)

### 🧪 Demo 8b – Scaling matters for KNN (Iris + a fake large-scale feature)

In [ ]:
Xi, yi = load_iris(return_X_y=True)
Xi_big = Xi.copy(); Xi_big[:, 0] *= 1000            # pretend sepal length is in micrometres (like Fare vs SibSp)
for name, model in [("no scaling", KNeighborsClassifier()), ("StandardScaler", make_pipeline(StandardScaler(), KNeighborsClassifier()))]:
    print(f"{name:15s} CV accuracy: {cross_val_score(model, Xi_big, yi, cv=5).mean():.3f}")

### 🧪 Demo 8c – Curse of dimensionality: distances concentrate

In [ ]:
rng = np.random.default_rng(RNG)
dims = [1, 2, 5, 10, 50, 100, 500, 1000]
ratios = []
for d in dims:
    pts = rng.uniform(size=(500, d)); q = rng.uniform(size=d)
    dist = np.linalg.norm(pts - q, axis=1)
    ratios.append((dist.max() - dist.min()) / dist.min())
plt.plot(dims, ratios, "o-"); plt.xscale("log"); plt.yscale("log")
plt.xlabel("number of features d"); plt.ylabel("(farthest − nearest) / nearest")
plt.title("As d grows, the 'nearest' neighbour is barely nearer than the farthest"); plt.show()

### 🧪 Demo 8d – Min-Max vs Standard vs Robust scaling with one outlier

In [ ]:
feat = np.array([10, 12, 11, 13, 12, 11, 500], dtype=float).reshape(-1, 1)
display(pd.DataFrame({"raw": feat.ravel(),
                      "MinMax": MinMaxScaler().fit_transform(feat).ravel(),
                      "Standard": StandardScaler().fit_transform(feat).ravel(),
                      "Robust": RobustScaler().fit_transform(feat).ravel()}).round(3))
print("MinMax squashes the normal values into ~0–0.006; Robust keeps them spread out.")

### 🏋️ Exercise 8.1 – Min-Max to a custom range, safe for constant features

In [ ]:
def min_max_scale(x, a=0.0, b=1.0):
    """Scale a 1-D array to [a, b]. If x is constant, return an array filled with a."""
    x = np.asarray(x, float)
    # TODO
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def min_max_scale(x, a=0.0, b=1.0):
    """Scale a 1-D array to [a, b]. If x is constant, return an array filled with a."""
    x = np.asarray(x, float)
    rng_ = x.max() - x.min()
    if rng_ == 0:
        return np.full_like(x, a)
    return a + (x - x.min()) * (b - a) / rng_
```

</details>

In [ ]:
sample = np.array([2, 4, 6, 10], dtype=float)
check("Ex 8.1 [0,1]", lambda: min_max_scale(sample), MinMaxScaler().fit_transform(sample.reshape(-1, 1)).ravel())
check("Ex 8.1 [-1,1]", lambda: min_max_scale(sample, -1, 1), MinMaxScaler((-1, 1)).fit_transform(sample.reshape(-1, 1)).ravel())
check("Ex 8.1 constant", lambda: min_max_scale([5, 5, 5], -1, 1), [-1, -1, -1])

### 🏋️ Exercise 8.2 – KNN from scratch (predict only)
Implement majority-vote KNN with Euclidean distance and compare it with sklearn on scaled Iris.

In [ ]:
def knn_predict(X_train, y_train, X_query, k=5):
    # TODO: for each query row: distances to all training rows, take k smallest (np.argsort), majority vote (np.bincount)
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def knn_predict(X_train, y_train, X_query, k=5):
    preds = []
    for q in X_query:
        d = np.sqrt(((X_train - q) ** 2).sum(axis=1))
        nearest = y_train[np.argsort(d)[:k]]
        preds.append(np.bincount(nearest).argmax())
    return np.array(preds)
```

</details>

In [ ]:
Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(Xi, yi, test_size=0.3, stratify=yi, random_state=RNG)
sc = StandardScaler().fit(Xtr_i)                       # fit on train only!
mine = knn_predict(sc.transform(Xtr_i), ytr_i, sc.transform(Xte_i), k=5)
ref = KNeighborsClassifier(5).fit(sc.transform(Xtr_i), ytr_i).predict(sc.transform(Xte_i))
check("Ex 8.2 agrees with sklearn on ≥ 95% of rows", lambda: None if mine is None else (mine == ref).mean() >= 0.95)

---
<a id="m9"></a>
# 🟡 Module 9 – Streaming Residual Alerts (Q12 refresher, Practical Lab 1)

### 📘 Recap
```
Robot controller ─► StreamingSimulator (1 reading / 2 s) ─► INSERT into Neon PostgreSQL (psycopg, creds from .env)
                                                               │
                         dashboard + alerts ◄── residual = actual − predicted ◄── LinearRegression (fit on healthy data)
```
- **Alert:** residual > μ + 2σ, **sustained** for ≥ T seconds → schedule maintenance.
- **Error:** residual > μ + 3σ, sustained → act now (avoid the 480-minute torque-tube downtime).
- **Duration** filters out one-off spikes. A longer T means higher precision but later detection (lower recall).
- `.env` holds `NEON_DATABASE_URL`, is read with `os.getenv`, and is listed in `.gitignore`.

### 🧪 Demo 9 – Synthetic axis current with a spike and a real fault (no database needed)

In [ ]:
rng = np.random.default_rng(RNG)
n = 900                                                  # 900 readings × 2 s = 30 min
t_s = np.arange(n) * 2.0
current = 5 + 0.0005 * t_s + rng.normal(0, 0.15, n)      # healthy behaviour with slow drift
current[300] += 1.5                                      # single transient spike (should NOT alert)
current[650:] += np.linspace(0.3, 1.2, n - 650)          # developing fault (SHOULD alert, then error)

healthy = slice(0, 250)
reg = LinearRegression().fit(t_s[healthy].reshape(-1, 1), current[healthy])
resid = current - reg.predict(t_s.reshape(-1, 1))
mu, sigma = resid[healthy].mean(), resid[healthy].std()
ALERT, ERROR = mu + 2 * sigma, mu + 3 * sigma
MIN_SAMPLES = 15                                         # 15 readings × 2 s = 30 s sustained

def sustained_naive(mask, min_len):
    """Reference implementation: True where mask has been True for >= min_len consecutive samples."""
    out, run = np.zeros_like(mask, bool), 0
    for i, m in enumerate(mask):
        run = run + 1 if m else 0
        out[i] = run >= min_len
    return out

alert = sustained_naive(resid > ALERT, MIN_SAMPLES)
error = sustained_naive(resid > ERROR, MIN_SAMPLES)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_s, resid, lw=0.8, label="residual")
ax.axhline(ALERT, c="orange", ls="--", label="alert threshold (μ+2σ)")
ax.axhline(ERROR, c="red", ls="--", label="error threshold (μ+3σ)")
ax.fill_between(t_s, resid.min(), resid.max(), where=alert, color="orange", alpha=0.15, label="ALERT active")
ax.fill_between(t_s, resid.min(), resid.max(), where=error, color="red", alpha=0.15, label="ERROR active")
ax.annotate("single spike → ignored", (600, resid[300]), (900, resid[300]), arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("time (s)"); ax.set_ylabel("residual (A)"); ax.legend(loc="upper left", fontsize=8); plt.show()

print(f"First ALERT at t = {t_s[alert.argmax()]:.0f} s, first ERROR at t = {t_s[error.argmax()]:.0f} s (fault starts at {t_s[650]:.0f} s)")

### 🏋️ Exercise 9.1 – Vectorize the duration rule
Write `sustained(mask, min_len)` **without** a Python loop over samples (hint: cumulative sums, or `np.convolve`). It must equal `sustained_naive`.

In [ ]:
def sustained(mask, min_len):
    mask = np.asarray(mask, bool)
    # TODO: vectorized run-length logic
    return None

<details>
<summary><b>💡 Show solution</b> (try it yourself first!)</summary>

```python
def sustained(mask, min_len):
    mask = np.asarray(mask, bool)
    window = np.convolve(mask.astype(int), np.ones(min_len, int), mode="full")[:len(mask)]
    return window >= min_len
```

</details>

In [ ]:
toy = np.array([0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0], bool)
check("Ex 9.1 toy example", lambda: None if sustained(toy, 3) is None else np.array_equal(sustained(toy, 3), sustained_naive(toy, 3)))
check("Ex 9.1 robot data", lambda: None if sustained(resid > ALERT, MIN_SAMPLES) is None
      else np.array_equal(sustained(resid > ALERT, MIN_SAMPLES), alert))

### 🏋️ Exercise 9.2 – Trade-off (written)
What happens to the false alarms and to detection delay if `MIN_SAMPLES` goes from 15 to 60? Connect your answer to precision and recall (Module 3).

> ✍️ *Your answer here*

---
<a id="behavioral"></a>
## Section 5 – Behavioral Answers (STAR) — Q13, Q14, Q15

**STAR + L:** **S**ituation (1 sentence) → **T**ask (your responsibility) → **A**ction (*I* did… with specifics, 60% of the answer) →
**R**esult (measurable) → **L**earning (what you now do differently). Aim for 60–90 seconds.

### Q13 – Peer code review (scored 3/4) → upgrade to 4/4
| Part | Your original | Upgrade to add |
|---|---|---|
| S/T | Peer programming between groups: load, EDA, preprocess | ✔ keep |
| A | Found EDA running before data loaded; reviewed the rest; showed evidence | **How you said it:** *"I asked: can we Restart & Run All to check this cell order?"* |
| R | Peer fixed it; notebook ran smoothly | **Why it mattered:** it only worked thanks to hidden kernel state, so it wasn't reproducible for grading |
| L | — | *"Now I always Restart & Run All before submitting or reviewing; we added it to our team checklist."* |

### Q14 – Leaked credentials (scored 2/4) → incident checklist
1. **Rotate the Neon password / create a new role immediately.** Once public, assume it's compromised.
2. Check the Neon activity / connection logs for unknown access.
3. `git rm --cached .env`, add `.env` to `.gitignore`, add `.env.example` with placeholders.
4. Purge history (`git filter-repo` / BFG), force-push, ask teammates to re-clone.
5. Notify the team (new secret via a secure channel) and the instructor / owner.
6. Blameless follow-up with the teammate plus prevention: a pre-commit secret scanner (`gitleaks`) and GitHub secret scanning.

### Q15 – Weak results & stakeholders (scored 1/4) → full structure
- **S/T:** "Our housing-price model had a lower R² than expected."
- **A (diagnose):** correlations with the target, scatter/residual plots, comparison with a `DummyRegressor` baseline, data-quality checks (caps, missing values, units).
- **A (iterate):** feature engineering / polynomial terms, a tree or KNN model, 5-fold CV, each run logged in `experiments/*.csv`.
- **R:** "R² rose from X to Y" *or* "no model beat about 0.5, which showed the limit was missing information (e.g., location), not the algorithm."
- **Stakeholder:** "It gets the direction right but is typically off by about $X. Use it to flag listings for review, not to set prices. Adding location data is the next step."
- **Correction:** PCA creates *combinations* of features. To find important original features, use correlations, `SelectKBest`, coefficients or tree importances.

### ✍️ Rewrite your three behavioral answers using the guidance above (practise saying them aloud)

**Q13 (STAR + L):**
> ✍️ *Your answer here*

**Q14:**
> ✍️ *Your answer here*

**Q15:**
> ✍️ *Your answer here*

---
<a id="requiz"></a>
## Section 6 – Re-Quiz (12 questions targeting your gaps)
Answer each one in the cell under the question **before** opening the answer. Score yourself out of 12, then record it in reflection **R7**.

**RQ1.** Your manager says: 'Use KNN to segment our unlabelled users into 4 groups.' What do you reply?

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

KNN is supervised and needs labels. For unlabelled segmentation, use clustering such as K-Means (k = 4, chosen or validated with elbow/silhouette), on scaled features.
</details>

**RQ2.** A column `size` ∈ {small, medium, large} and a column `city` ∈ {Toronto, Waterloo, Kitchener}. How do you encode each for a KNN model?

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

`size` is ordinal → OrdinalEncoder with categories=[['small','medium','large']] (then scale). `city` is nominal → OneHotEncoder(handle_unknown='ignore'). Both go inside a ColumnTransformer in the Pipeline.
</details>

**RQ3.** You standardize the whole dataset, then run 5-fold CV. What's wrong, and what is the one-line fix?

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

The scaler's mean/std include the validation rows of every fold → leakage. Put the scaler in a `Pipeline` and pass the pipeline to `cross_val_score`.
</details>

**RQ4.** A KNN with k = 1 has train accuracy 1.00 and CV accuracy 0.71. Diagnose it and name two fixes.

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

High variance (overfitting): large train–validation gap. Increase k (tune with CV), scale or select features, get more data.
</details>

**RQ5.** Your fraud model has 99.5% accuracy on data with 0.5% fraud. What two numbers do you ask for next?

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

Recall (and precision) on the fraud class, plus the DummyClassifier baseline (≈99.5%). The model may catch zero fraud.
</details>

**RQ6.** Compute precision and recall: TP = 30, FP = 10, FN = 20.

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

Precision = 30/40 = 0.75; recall = 30/50 = 0.60; F1 = 2·0.75·0.6/1.35 ≈ 0.667.
</details>

**RQ7.** Your gradient-descent loss goes 10 → 40 → 300 → inf. Cause and fix?

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

Learning rate too large (and/or unscaled features). Lower α by ~10×, standardize features, plot loss vs epoch.
</details>

**RQ8.** Write ∂J/∂θ₁ for J = (1/2m)Σ(θ₀ + θ₁x − y)².

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

(1/m)Σ(θ₀ + θ₁xᵢ − yᵢ)·xᵢ. The xᵢ comes from the chain rule.
</details>

**RQ9.** Test R² = −0.2. What does it mean?

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

The model is worse on test data than always predicting the training mean. Likely a badly overfit or broken model, or distribution shift.
</details>

**RQ10.** Split x=[1,2,3,4,5,6], y=[1,1,1,9,9,9]: best threshold and leaf values?

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

3.5. Children are pure (weighted MSE = 0); leaves predict 1 and 9.
</details>

**RQ11.** Write the sigmoid and explain why the logistic-regression decision boundary is linear.

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

σ(z) = 1/(1+e^(−z)). p ≥ 0.5 ⇔ z = w·x + b ≥ 0, which is a hyperplane.
</details>

**RQ12.** You need to validate a rainfall-prediction model on 5 years of daily data. Which CV and why?

> ✍️ *Your answer here*

<details><summary>Show answer</summary>

TimeSeriesSplit / forward-chaining (or a chronological holdout): train on the past, validate on the future, never shuffle. Lag features must only use past values.
</details>

In [ ]:
# Record your re-quiz score here and compare with the original interview's technical score
requiz_score = None          # TODO: e.g. 9  (out of 12)
original_tech_pct = quiz.loc[quiz.type == "Technical", "score"].sum() / (4 * (quiz.type == "Technical").sum())
if requiz_score is not None:
    print(f"Original technical: {original_tech_pct:.0%}  →  Re-quiz: {requiz_score / 12:.0%}")
else:
    print(f"Original technical: {original_tech_pct:.0%}. Set requiz_score after completing Section 6.")

---
<a id="submit"></a>
## ✅ Submission Checklist
- [ ] All **Reflection** cells (R1–R7) completed in your own words
- [ ] Exercises attempted. `check()` cells show ✅ (keep your own code, not just the revealed solutions)
- [ ] Written exercises (1.3, 4.2, 5.2, 9.2) and behavioral rewrites completed
- [ ] Re-quiz answered and `requiz_score` recorded
- [ ] **Kernel → Restart & Run All** completes without errors (lesson from Q13!)
- [ ] Only `JobInterviewGuide_Workshop.ipynb` is committed and pushed to GitHub (no `.env`, no data files)

```bash
git add JobInterviewGuide_Workshop.ipynb
git commit -m "Add personalized Job Interview Guide workshop notebook"
git push
```